# HKU VIIRS Monthly Nighttime Lights × Bolivia Municipalities

Favio Leiva

Powered by GPT 5.5

Notebook draft for calculating monthly local/zonal statistics by Bolivian municipality using `faviolc/hku-viirs-nighttime-lights-global` and the optimized Bolivia municipal map from `quarcs-lab/ds4bolivia`.

This notebook computes monthly zonal statistics for Bolivian municipalities using the global monthly VIIRS-like nighttime lights dataset stored on Hugging Face.

Processing strategy:

* Outer loop: process one year at a time.
* Inner loop: process monthly rasters in parallel within that year.
* After each year: save a yearly checkpoint and delete temporary raster files.
* Final output: one row per municipality × year × month.

Expected final structure:

* `ID_ADMIN`
* `asdf_id`
* `mun_id`
* `dep_id`
* `mun_name`
* `dep_name`
* `year`
* `month`
* `count`
* `min`
* `max`
* `median`
* `mean`
* `sum`
* `std`
* `count_zeros`
* `count_positive`
* `count_invalid`

The raster repository is public, so no Hugging Face token is required for the nighttime lights data. The Bolivia vector layer is also public and is loaded directly from the `quarcs-lab/ds4bolivia` GitHub repository, using the optimized municipal map:

`maps/bolivia339geoqueryOpt.geojson`

For the Bolivia municipal map, `asdf_id` is used as the source municipality identifier and standardized internally as `ID_ADMIN`. The notebook also preserves available municipality and department attributes, such as `mun_id`, `dep_id`, `mun_name`, and `dep_name`, when available from the optimized map or from the accompanying region-name metadata.

The workflow is designed to be restartable: if a runtime stops, existing yearly checkpoints are skipped automatically when the production loop is rerun.

The expected unit of analysis is Bolivia’s 339 municipalities. Therefore, the notebook includes validation checks confirming that the optimized municipal map contains 339 features and 339 unique `ID_ADMIN` values before running the full production loop.

A smoke test is preserved before the full run. For 1992, the expected smoke-test output is:

* 339 municipalities × 9 available months = 3,051 rows

because the nighttime lights series starts in April 1992.

The optimized map is used as the default production vector layer because it is lighter and faster for repeated zonal-statistics operations. A robustness check can optionally compare one month or one year against `bolivia339geoqueryFull.geojson` to verify whether boundary simplification materially affects municipal nighttime-light statistics.


## Step 1. Installations

In [ ]:
!pip -q install huggingface_hub    # Interact with Hugging Face Hub (models, datasets, spaces)
!pip -q install geopandas          # Work with geospatial data in pandas DataFrames
!pip -q install pyogrio            # Fast I/O for geospatial vector data
!pip -q install rasterstats        # Zonal statistics for raster data
!pip -q install rasterio           # Read and write geospatial raster data
!pip -q install pandas             # Data manipulation and analysis
!pip -q install numpy              # Numerical computing
!pip -q install joblib             # Parallel processing and caching
!pip -q install psutil             # System and process utilities
!pip -q install matplotlib         # Plotting and visualization
!pip -q install pyproj             # Cartographic projections and coordinate transformations
!pip -q install shapely            # Manipulate and analyze geometric objects
!pip -q install pyarrow            # Apache Arrow for columnar data

## Step 2. Imports and settings

In [ ]:
# ------------------------------------------------------------------------------
# Thread control
# ------------------------------------------------------------------------------
# We parallelize at the month level using joblib. To avoid accidental
# over-parallelization inside GDAL/BLAS/OpenMP, we force most internal libraries
# to use one thread per worker process.

import os

os.environ["GDAL_NUM_THREADS"] = "1"      # Restrict GDAL to a single thread
os.environ["OMP_NUM_THREADS"] = "1"       # Restrict OpenMP to a single thread
os.environ["MKL_NUM_THREADS"] = "1"       # Restrict Intel MKL to a single thread
os.environ["OPENBLAS_NUM_THREADS"] = "1"  # Restrict OpenBLAS to a single thread
os.environ["NUMEXPR_NUM_THREADS"] = "1"   # Restrict NumExpr to a single thread

# ------------------------------------------------------------------------------
# Imports
# ------------------------------------------------------------------------------

import re            # Regular expression operations
import gc            # Garbage Collector interface
import json          # JSON encoder and decoder
import shutil        # High-level file operations
import zipfile       # Work with ZIP archives
import traceback     # Print or retrieve a stack traceback
import multiprocessing # Process-based parallelism
from pathlib import Path # Object-oriented filesystem paths

import psutil        # Process and system utilities (CPU, memory, disks, network)
import numpy as np   # Numerical computing library
import pandas as pd  # Data manipulation and analysis library
import geopandas as gpd # Geographic data in pandas DataFrames
import rasterio      # Read and write geospatial raster datasets
import matplotlib.pyplot as plt # Plotting library

from pyproj import CRS # Coordinate Reference System objects
from rasterstats import zonal_stats # Zonal statistics for raster data
from huggingface_hub import HfApi, hf_hub_download # Hugging Face Hub API and file downloading
from joblib import Parallel, delayed # Parallel computing with joblib

# ------------------------------------------------------------------------------
# Hardware check
# ------------------------------------------------------------------------------

total_cores = multiprocessing.cpu_count()
total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
available_ram_gb = psutil.virtual_memory().available / (1024 ** 3)

print("--------------------------------------------------")
print(f"CPU logical cores available : {total_cores}")
print(f"Total RAM available         : {total_ram_gb:.2f} GB")
print(f"Currently available RAM     : {available_ram_gb:.2f} GB")
print("--------------------------------------------------")

if total_cores >= 32:
    print("Many-core runtime detected. Month-level parallelism should be useful.")
elif total_cores >= 12:
    print("Moderate-core runtime detected. Use balanced month-level parallelism.")
else:
    print("Limited CPU cores detected. Use safe mode.")

if total_ram_gb >= 50:
    print("High-RAM runtime detected.")
elif total_ram_gb >= 25:
    print("Medium/high RAM runtime detected.")
else:
    print("Standard RAM detected. Use fewer monthly workers.")

## Step 3. Zonal statistics and parallelism

In [ ]:
# ------------------------------------------------------------------------------
# Country and period
# ------------------------------------------------------------------------------

COUNTRY_NAME = "Bolivia"
COUNTRY_SLUG = "bolivia"
COUNTRY_PREFIX = "BOL"

START_YEAR = 1992
END_YEAR = 2024

PROJECT_NAME = f"HKU_VIIRS_Monthly_{COUNTRY_NAME}"
PRODUCT_NAME = "HKU_VIIRS_Monthly"

# The dataset starts in April 1992.
FIRST_AVAILABLE_MONTH_BY_YEAR = {
    1992: 4
}

# Expected number of Bolivian municipalities in ds4bolivia.
EXPECTED_UNITS = 339

# ------------------------------------------------------------------------------
# Raster repository: monthly HKU VIIRS workflow
# ------------------------------------------------------------------------------

RASTER_REPO_ID = "faviolc/hku-viirs-nighttime-lights-global"
RASTER_REPO_TYPE = "dataset"
RASTER_BASE_FOLDER = "raw"
YEAR_ZIP_TEMPLATE = "raw/{year}/{year}.zip"

# ------------------------------------------------------------------------------
# Bolivia vector source: ds4bolivia optimized municipal map
# ------------------------------------------------------------------------------
# For Bolivia, we do not use a private Hugging Face shapefile.
# We load the optimized municipal GeoJSON directly from GitHub.

VECTOR_REPO_URL = "https://raw.githubusercontent.com/quarcs-lab/ds4bolivia/master"
VECTOR_MAP_FOLDER = "maps"
VECTOR_FILENAME = "bolivia339geoqueryOpt.geojson"
VECTOR_URL = f"{VECTOR_REPO_URL}/{VECTOR_MAP_FOLDER}/{VECTOR_FILENAME}"

# Optional metadata table with municipality and department names.
REGION_NAMES_URL = f"{VECTOR_REPO_URL}/regionNames/regionNames.csv"

# Source ID in ds4bolivia. This will be standardized later as ID_ADMIN.
SOURCE_ID_COL = "asdf_id"
ID_COL = "ID_ADMIN"

# No vector token is required for ds4bolivia.
VECTOR_TOKEN_SECRET_NAME = None

# ------------------------------------------------------------------------------
# Zonal statistics
# ------------------------------------------------------------------------------

ZONAL_STATS = ["count", "min", "max", "median", "mean", "sum", "std"]

def count_zeros_stat(values):
    """
    Count valid pixels equal to zero.
    """
    if np.ma.isMaskedArray(values):
        arr = values.compressed()
    else:
        arr = np.asarray(values).ravel()

    arr = arr[np.isfinite(arr)]
    return int(np.sum(arr == 0))


def count_positive_stat(values):
    """
    Count valid pixels greater than zero.
    """
    if np.ma.isMaskedArray(values):
        arr = values.compressed()
    else:
        arr = np.asarray(values).ravel()

    arr = arr[np.isfinite(arr)]
    return int(np.sum(arr > 0))


def count_invalid_stat(values):
    """
    Count masked or non-finite pixels.

    Note:
    In rasterstats, masked values may include nodata pixels and pixels outside
    the polygon window. We use this mainly as a diagnostic and verify it during
    the smoke test.
    """
    if np.ma.isMaskedArray(values):
        mask = np.ma.getmaskarray(values)
        data = np.asarray(values.data)

        masked_count = int(np.sum(mask))
        nonfinite_unmasked_count = int(np.sum((~mask) & (~np.isfinite(data))))

        return masked_count + nonfinite_unmasked_count

    arr = np.asarray(values).ravel()
    return int(np.sum(~np.isfinite(arr)))


CUSTOM_ZONAL_STATS = {
    "count_zeros": count_zeros_stat,
    "count_positive": count_positive_stat,
    "count_invalid": count_invalid_stat
}

ALL_ZONAL_OUTPUT_STATS = ZONAL_STATS + list(CUSTOM_ZONAL_STATS.keys())

ALL_TOUCHED = False

# ------------------------------------------------------------------------------
# Parallelism mode
# ------------------------------------------------------------------------------
# We process years sequentially, but months inside each year in parallel.
# For Bolivia, start in safe mode and increase only after the smoke test works.
# Bolivia has fewer units than Peru, but memory use is still driven by raster size.

PROCESSING_MODE = "safe"  # "safe", "balanced", or "aggressive"

MODE_SETTINGS = {
    "safe": {
        "ram_per_worker_gb": 5.0,
        "max_month_workers": 4
    },
    "balanced": {
        "ram_per_worker_gb": 3.5,
        "max_month_workers": 6
    },
    "aggressive": {
        "ram_per_worker_gb": 2.5,
        "max_month_workers": 8
    }
}

if PROCESSING_MODE not in MODE_SETTINGS:
    raise ValueError(f"Invalid PROCESSING_MODE: {PROCESSING_MODE}")

# ------------------------------------------------------------------------------
# Output paths
# ------------------------------------------------------------------------------
# Important: these variables must be defined before printing them.

BASE_DIR = Path("hku_viirs_bolivia_work")
TEMP_DIR = BASE_DIR / "temp"
VECTOR_DIR = BASE_DIR / "vector"
CHECKPOINT_DIR = BASE_DIR / "checkpoints_yearly"
OUTPUT_DIR = BASE_DIR / "outputs"

for folder in [BASE_DIR, TEMP_DIR, VECTOR_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / f"{PROJECT_NAME}_Admin_Stats_{START_YEAR}_{END_YEAR}.csv"
OUTPUT_PARQUET = OUTPUT_DIR / f"{PROJECT_NAME}_Admin_Stats_{START_YEAR}_{END_YEAR}.parquet"
OUTPUT_ZIP = OUTPUT_DIR / f"{PROJECT_NAME}_Admin_Stats_{START_YEAR}_{END_YEAR}.zip"

print("Configuration loaded.")
print(f"Country           : {COUNTRY_NAME}")
print(f"Product           : {PRODUCT_NAME}")
print(f"Raster repo       : {RASTER_REPO_ID}")
print(f"Raster pattern    : {YEAR_ZIP_TEMPLATE}")
print(f"Vector source     : GitHub / ds4bolivia")
print(f"Vector URL        : {VECTOR_URL}")
print(f"Region names URL  : {REGION_NAMES_URL}")
print(f"Source ID column  : {SOURCE_ID_COL}")
print(f"Internal ID column: {ID_COL}")
print(f"Expected units    : {EXPECTED_UNITS}")
print(f"Years             : {START_YEAR}-{END_YEAR}")
print(f"Zonal stats       : {ALL_ZONAL_OUTPUT_STATS}")
print(f"Processing mode   : {PROCESSING_MODE}")
print(f"Base directory    : {BASE_DIR}")
print(f"Temp directory    : {TEMP_DIR}")
print(f"Vector directory  : {VECTOR_DIR}")
print(f"Checkpoint dir    : {CHECKPOINT_DIR}")
print(f"Output CSV        : {OUTPUT_CSV}")
print(f"Output Parquet    : {OUTPUT_PARQUET}")
print(f"Output ZIP        : {OUTPUT_ZIP}")

## Step 4. Hugging Face API

In [ ]:
# ------------------------------------------------------------------------------
# Hugging Face API setup
# ------------------------------------------------------------------------------
# The raster repository is public, so we use no token for raster downloads.
# The Bolivia vector layer is also public and is loaded directly from GitHub:
# maps/bolivia339geoqueryOpt.geojson
#
# Therefore:
# - no Hugging Face token is required for rasters
# - no Hugging Face token is required for the Bolivia vector map
# - no private vector repository is used

RASTER_TOKEN = None
VECTOR_HF_TOKEN = None

# Hugging Face API object for public raster dataset access.
raster_api = HfApi()

# No vector_api is needed for Bolivia because the vector file is read from GitHub.
vector_api = None

print("Hugging Face access configured.")
print("Raster repository access: public/no token")
print("Vector repository access: public GitHub/no token")
print(f"Raster repo             : {RASTER_REPO_ID}")
print(f"Raster pattern          : {YEAR_ZIP_TEMPLATE}")
print(f"Vector source           : GitHub / ds4bolivia")
print(f"Vector URL              : {VECTOR_URL}")
print(f"Region names URL        : {REGION_NAMES_URL}")
print(f"Vector file             : {VECTOR_FILENAME}")
print(f"Source ID column        : {SOURCE_ID_COL}")
print(f"Internal ID column      : {ID_COL}")

## Step 5. Loading map

In [ ]:
# ------------------------------------------------------------------------------
# Cell 5 — Download and load Bolivia administrative GeoJSON
# ------------------------------------------------------------------------------
# This cell downloads the public Bolivia optimized municipal boundary file from
# the quarcs-lab/ds4bolivia GitHub repository.
#
# Expected vector file:
# - maps/bolivia339geoqueryOpt.geojson
#
# The file is expected to contain 339 Bolivian municipalities.
# No Hugging Face vector token is required.

print(f"Fetching {COUNTRY_NAME} administrative boundaries from GitHub / ds4bolivia...")
print(f"Vector URL: {VECTOR_URL}")

# ------------------------------------------------------------------------------
# Download vector file locally
# ------------------------------------------------------------------------------

import requests

vector_path = VECTOR_DIR / VECTOR_FILENAME

response = requests.get(VECTOR_URL, timeout=120)
response.raise_for_status()

with open(vector_path, "wb") as f:
    f.write(response.content)

print("--------------------------------------------------")
print(f"Downloaded vector file : {vector_path}")
print(f"File size              : {vector_path.stat().st_size / (1024 ** 2):.2f} MB")
print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Load vector file
# ------------------------------------------------------------------------------
# pyogrio is faster. If it fails, we fall back to the default GeoPandas engine.

try:
    gdf_admin = gpd.read_file(vector_path, engine="pyogrio")
except Exception as e:
    print("pyogrio read failed. Falling back to default GeoPandas engine.")
    print(e)
    gdf_admin = gpd.read_file(vector_path)

print("--------------------------------------------------")
print(f"Raw polygons/features loaded : {len(gdf_admin):,}")
print(f"Expected features            : {EXPECTED_UNITS:,}")
print(f"Original vector CRS          : {gdf_admin.crs}")
print(f"Geometry column              : {gdf_admin.geometry.name}")
print("Original columns:")
display(gdf_admin.head())

# ------------------------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------------------------

if len(gdf_admin) != EXPECTED_UNITS:
    print(
        f"Warning: expected {EXPECTED_UNITS:,} Bolivian municipalities, "
        f"but loaded {len(gdf_admin):,} features."
    )

if SOURCE_ID_COL not in gdf_admin.columns:
    raise ValueError(
        f"Expected source ID column '{SOURCE_ID_COL}' was not found in the Bolivia GeoJSON.\n"
        f"Available columns: {list(gdf_admin.columns)}"
    )

print("--------------------------------------------------")
print("Bolivia administrative boundaries loaded successfully.")
print(f"Source ID column found: {SOURCE_ID_COL}")
print(f"Unique source IDs     : {gdf_admin[SOURCE_ID_COL].nunique():,}")
print("--------------------------------------------------")

In [ ]:
# ------------------------------------------------------------------------------
# Inspect Bolivia municipal GeoJSON columns
# ------------------------------------------------------------------------------

print("Shape:", gdf_admin.shape)
print("Expected units:", EXPECTED_UNITS)
print("CRS:", gdf_admin.crs)
print("Geometry column:", gdf_admin.geometry.name)

columns_table = pd.DataFrame({
    "column": gdf_admin.columns,
    "dtype": [str(gdf_admin[col].dtype) for col in gdf_admin.columns],
    "non_null": [gdf_admin[col].notna().sum() for col in gdf_admin.columns],
    "unique_values": [
        gdf_admin[col].nunique(dropna=True) if col != gdf_admin.geometry.name else None
        for col in gdf_admin.columns
    ]
})

display(columns_table)

print("--------------------------------------------------")
print(f"Source ID column       : {SOURCE_ID_COL}")
print(f"Unique source IDs      : {gdf_admin[SOURCE_ID_COL].nunique():,}")
print(f"Expected municipalities: {EXPECTED_UNITS:,}")
print("--------------------------------------------------")

display(gdf_admin.head())

In [ ]:
# ------------------------------------------------------------------------------
# Feature selection and standardization — Bolivia municipal GeoJSON
# ------------------------------------------------------------------------------
# The Bolivia vector layer comes from quarcs-lab/ds4bolivia:
#
# - maps/bolivia339geoqueryOpt.geojson
#
# The stable source identifier is:
#
# - asdf_id : municipality identifier used across ds4bolivia datasets
#
# We standardize asdf_id as ID_ADMIN, which will be used as the stable identifier
# in the final municipality-year-month output.
#
# If available, the notebook also preserves municipality and department metadata,
# such as mun_id, mun_name, dep_id, and dep_name.

def first_existing(candidates, attributes):
    for candidate in candidates:
        if candidate in attributes:
            return candidate
    return None


geometry_col = gdf_admin.geometry.name
non_geometry_cols = [c for c in gdf_admin.columns if c != geometry_col]

# ------------------------------------------------------------------------------
# Optional enrichment with ds4bolivia region names
# ------------------------------------------------------------------------------
# The optimized GeoJSON may already contain names. If not, we try to merge the
# regionNames table using asdf_id. This keeps the notebook robust.

try:
    print("Trying to load optional Bolivia region-name metadata...")
    df_region_names = pd.read_csv(REGION_NAMES_URL)

    if SOURCE_ID_COL in gdf_admin.columns and SOURCE_ID_COL in df_region_names.columns:
        gdf_admin[SOURCE_ID_COL] = (
            gdf_admin[SOURCE_ID_COL]
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )

        df_region_names[SOURCE_ID_COL] = (
            df_region_names[SOURCE_ID_COL]
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )

        # Avoid overwriting columns already present in the GeoJSON.
        new_region_cols = [
            c for c in df_region_names.columns
            if c == SOURCE_ID_COL or c not in gdf_admin.columns
        ]

        gdf_admin = gdf_admin.merge(
            df_region_names[new_region_cols],
            on=SOURCE_ID_COL,
            how="left"
        )

        print("Region-name metadata merged successfully.")
    else:
        print("Region-name metadata not merged because asdf_id was not found in both tables.")

except Exception as e:
    print("Optional region-name metadata could not be loaded or merged.")
    print(e)

# Refresh columns after optional merge
geometry_col = gdf_admin.geometry.name
non_geometry_cols = [c for c in gdf_admin.columns if c != geometry_col]

# ------------------------------------------------------------------------------
# Bolivia-specific candidate columns
# ------------------------------------------------------------------------------

id_candidates = [
    "asdf_id", "ASDF_ID",
    SOURCE_ID_COL,
    "id", "ID"
]

department_code_candidates = [
    "dep_id", "DEP_ID",
    "depto_id", "DEPTO_ID",
    "department_id", "DEPARTMENT_ID",
    "cod_dep", "COD_DEP"
]

department_name_candidates = [
    "dep_name", "DEP_NAME",
    "depto_name", "DEPTO_NAME",
    "department_name", "DEPARTMENT_NAME",
    "departamento", "DEPARTAMENTO",
    "depto", "DEPTO",
    "NAME_1", "ADM1_NAME"
]

municipality_code_candidates = [
    "mun_id", "MUN_ID",
    "municipality_id", "MUNICIPALITY_ID",
    "municipio_id", "MUNICIPIO_ID",
    "cod_mun", "COD_MUN"
]

municipality_name_candidates = [
    "mun_name", "MUN_NAME",
    "municipality_name", "MUNICIPALITY_NAME",
    "municipality", "MUNICIPALITY",
    "municipio", "MUNICIPIO",
    "name", "NAME",
    "NAME_2", "ADM2_NAME"
]

SOURCE_ID_COL = first_existing(id_candidates, non_geometry_cols)

DEPARTMENT_CODE_COL = first_existing(department_code_candidates, non_geometry_cols)
DEPARTMENT_COL = first_existing(department_name_candidates, non_geometry_cols)

MUNICIPALITY_CODE_COL = first_existing(municipality_code_candidates, non_geometry_cols)
NAME_COL = first_existing(municipality_name_candidates, non_geometry_cols)

print("--------------------------------------------------")
print(f"Detected source ID column          : {SOURCE_ID_COL}")
print(f"Detected department code column   : {DEPARTMENT_CODE_COL}")
print(f"Detected department name column   : {DEPARTMENT_COL}")
print(f"Detected municipality code column : {MUNICIPALITY_CODE_COL}")
print(f"Detected municipality name column : {NAME_COL}")
print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Create standardized ID_ADMIN
# ------------------------------------------------------------------------------

if SOURCE_ID_COL is None:
    gdf_admin["ID_ADMIN"] = [f"{COUNTRY_PREFIX}_{i + 1:04d}" for i in range(len(gdf_admin))]
    print("No source ID column found. Created ID_ADMIN from row order.")
else:
    source_id = (
        gdf_admin[SOURCE_ID_COL]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    missing_like = (
        source_id.isna()
        | source_id.eq("")
        | source_id.str.lower().isin(["nan", "none", "null"])
    )

    duplicated = source_id.duplicated(keep=False)

    if missing_like.any() or duplicated.any():
        gdf_admin["ID_ADMIN"] = (
            source_id.where(~missing_like, "missing")
            + "_"
            + gdf_admin.index.astype(str)
        )
        print(f"Source ID column '{SOURCE_ID_COL}' was incomplete or not unique.")
        print("Created unique ID_ADMIN by appending the row index.")
    else:
        gdf_admin["ID_ADMIN"] = source_id
        print(f"Created ID_ADMIN from source column: {SOURCE_ID_COL}")

ID_COL = "ID_ADMIN"

# Keep the expected value from the configuration cell.
# For ds4bolivia, this should be 339.
if "EXPECTED_UNITS" not in globals():
    EXPECTED_UNITS = 339

# ------------------------------------------------------------------------------
# Preserve useful attributes for the final output
# ------------------------------------------------------------------------------

priority_cols = [
    "ID_ADMIN",
    SOURCE_ID_COL,
    DEPARTMENT_CODE_COL,
    DEPARTMENT_COL,
    MUNICIPALITY_CODE_COL,
    NAME_COL
]

priority_cols = [
    c for c in priority_cols
    if c is not None and c in gdf_admin.columns
]

ATTRIBUTE_COLS = []

for col in priority_cols + non_geometry_cols:
    if col in gdf_admin.columns and col != geometry_col and col not in ATTRIBUTE_COLS:
        ATTRIBUTE_COLS.append(col)

# Ensure ID_ADMIN is first
ATTRIBUTE_COLS = ["ID_ADMIN"] + [c for c in ATTRIBUTE_COLS if c != "ID_ADMIN"]

print("Administrative columns selected:")
print(f"ID_COL                  : {ID_COL}")
print(f"SOURCE_ID_COL           : {SOURCE_ID_COL}")
print(f"DEPARTMENT_CODE_COL     : {DEPARTMENT_CODE_COL}")
print(f"DEPARTMENT_COL          : {DEPARTMENT_COL}")
print(f"MUNICIPALITY_CODE_COL   : {MUNICIPALITY_CODE_COL}")
print(f"NAME_COL                : {NAME_COL}")
print(f"EXPECTED_UNITS          : {EXPECTED_UNITS:,}")
print(f"ATTRIBUTE_COLS          : {len(ATTRIBUTE_COLS)} columns")
print(ATTRIBUTE_COLS)

# ------------------------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------------------------

n_features = len(gdf_admin)
n_unique_ids = gdf_admin[ID_COL].nunique(dropna=True)
n_duplicate_ids = int(gdf_admin[ID_COL].duplicated().sum())

print("--------------------------------------------------")
print(f"Rows/features          : {n_features:,}")
print(f"Expected municipalities: {EXPECTED_UNITS:,}")
print(f"Unique ID_ADMIN        : {n_unique_ids:,}")
print(f"Duplicated IDs         : {n_duplicate_ids:,}")
print("--------------------------------------------------")

if n_features != EXPECTED_UNITS:
    print(
        f"Warning: expected {EXPECTED_UNITS:,} Bolivian municipalities, "
        f"but found {n_features:,} features."
    )

if n_duplicate_ids > 0:
    print("Warning: ID_ADMIN has duplicates. Review SOURCE_ID_COL before continuing.")
else:
    print("Good: ID_ADMIN is unique.")

if n_unique_ids != EXPECTED_UNITS:
    print(
        f"Warning: expected {EXPECTED_UNITS:,} unique ID_ADMIN values, "
        f"but found {n_unique_ids:,}."
    )
else:
    print("Good: unique ID_ADMIN count matches expected Bolivia municipalities.")

# ------------------------------------------------------------------------------
# Small preview without geometry
# ------------------------------------------------------------------------------

preview_cols = [
    "ID_ADMIN",
    SOURCE_ID_COL,
    DEPARTMENT_CODE_COL,
    DEPARTMENT_COL,
    MUNICIPALITY_CODE_COL,
    NAME_COL
]

preview_cols = [
    c for c in preview_cols
    if c is not None and c in gdf_admin.columns
]

preview_cols = list(dict.fromkeys(preview_cols))

display(gdf_admin[preview_cols].head())

In [ ]:
# ------------------------------------------------------------------------------
# Cell 7 — Visual check of Bolivia municipal polygons
# ------------------------------------------------------------------------------
# This cell plots the cleaned Bolivia municipal polygons and prints basic
# spatial metadata. It does not modify the data.

print(f"Visual check of {COUNTRY_NAME} municipal polygons...")

print("--------------------------------------------------")
print(f"Features           : {len(gdf_admin):,}")
print(f"CRS                : {gdf_admin.crs}")
print(f"Geometry column    : {gdf_admin.geometry.name}")
print(f"Total bounds       : {gdf_admin.total_bounds}")
print(f"ID column          : {ID_COL}")
print(f"Unique ID_ADMIN    : {gdf_admin[ID_COL].nunique():,}")

if "EXPECTED_UNITS" in globals():
    print(f"Expected units     : {EXPECTED_UNITS:,}")

print("--------------------------------------------------")

# Check duplicated IDs before plotting
n_duplicate_ids = int(gdf_admin[ID_COL].duplicated().sum())

if n_duplicate_ids > 0:
    print(f"Warning: duplicated ID_ADMIN values found: {n_duplicate_ids:,}")
else:
    print("Good: ID_ADMIN is unique.")

# Optional: print useful Bolivia administrative columns if available
useful_cols = [
    "ID_ADMIN",
    "asdf_id",
    "mun_id",
    "mun_name",
    "dep_id",
    "dep_name",
    "municipio",
    "departamento",
    "name",
    "NAME"
]

useful_cols = [c for c in useful_cols if c in gdf_admin.columns]

if useful_cols:
    print("Preview of administrative attributes:")
    display(gdf_admin[useful_cols].head())

# Plot polygons
fig, ax = plt.subplots(figsize=(10, 12))

gdf_admin.plot(
    ax=ax,
    color="none",
    edgecolor="black",
    linewidth=0.15
)

ax.set_title(
    f"{COUNTRY_NAME} municipal polygons: {len(gdf_admin):,} units",
    fontsize=16,
    pad=12
)

ax.set_axis_off()
plt.tight_layout()
plt.show()

## Step 6. Revising Public Raster Repository

In [ ]:
# ------------------------------------------------------------------------------
# Cell 8 — List annual ZIP files in the public HKU VIIRS raster repository
# ------------------------------------------------------------------------------
# This cell checks the public raster repository and identifies the annual ZIP
# files to process, following the expected pattern:
#
# raw/1992/1992.zip
# raw/1993/1993.zip
# ...
#
# This cell is country-neutral for rasters. The Bolivia-specific vector file
# should already be loaded in previous cells from:
#
# quarcs-lab/ds4bolivia/maps/bolivia339geoqueryOpt.geojson

print("Listing annual ZIP files in the public HKU VIIRS raster repository...")

# ------------------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------------------

required_config_objects = [
    "COUNTRY_NAME",
    "RASTER_REPO_ID",
    "RASTER_REPO_TYPE",
    "RASTER_BASE_FOLDER",
    "YEAR_ZIP_TEMPLATE",
    "START_YEAR",
    "END_YEAR",
    "raster_api",
    "gdf_admin",
    "ID_COL"
]

missing_config_objects = [
    obj for obj in required_config_objects
    if obj not in globals()
]

if missing_config_objects:
    raise RuntimeError(
        "Missing required configuration objects: "
        + ", ".join(missing_config_objects)
    )

if "Tang-LRCC-DVNL" in RASTER_REPO_ID or "LACC" in RASTER_REPO_ID:
    raise ValueError(
        "The raster repository is still pointing to the Tang/LACC workflow. "
        "For this notebook, use the HKU VIIRS monthly raster repository."
    )

if COUNTRY_NAME.lower() != "bolivia":
    print(
        f"Warning: COUNTRY_NAME is currently '{COUNTRY_NAME}'. "
        "This notebook section is configured for the Bolivia case."
    )

if ID_COL not in gdf_admin.columns:
    raise ValueError(
        f"ID column '{ID_COL}' was not found in gdf_admin. "
        "Run the Bolivia feature-standardization cell before this one."
    )

print("Raster configuration")
print("-" * 70)
print(f"Country            : {COUNTRY_NAME}")
print(f"Raster repo        : {RASTER_REPO_ID}")
print(f"Raster repo type   : {RASTER_REPO_TYPE}")
print(f"Raster base folder : {RASTER_BASE_FOLDER}")
print(f"Year ZIP pattern   : {YEAR_ZIP_TEMPLATE}")
print(f"Target years       : {START_YEAR}-{END_YEAR}")
print("-" * 70)

print("Bolivia vector configuration")
print("-" * 70)
print(f"Vector features    : {len(gdf_admin):,}")
print(f"Unique ID_ADMIN    : {gdf_admin[ID_COL].nunique():,}")

if "EXPECTED_UNITS" in globals():
    print(f"Expected units     : {EXPECTED_UNITS:,}")

if "VECTOR_URL" in globals():
    print(f"Vector URL         : {VECTOR_URL}")

print("-" * 70)

# ------------------------------------------------------------------------------
# List files from public raster repository
# ------------------------------------------------------------------------------

raster_files = raster_api.list_repo_files(
    repo_id=RASTER_REPO_ID,
    repo_type=RASTER_REPO_TYPE
)

print(f"Total files in raster repository: {len(raster_files):,}")

# ------------------------------------------------------------------------------
# Identify annual ZIP files
# ------------------------------------------------------------------------------
# Expected structure:
# raw / YYYY / YYYY.zip

zip_records = []

for file_path in raster_files:
    parts = file_path.split("/")

    if len(parts) != 3:
        continue

    folder, folder_year_text, zip_name = parts

    if folder != RASTER_BASE_FOLDER:
        continue

    if not folder_year_text.isdigit():
        continue

    if not zip_name.lower().endswith(".zip"):
        continue

    file_year_text = zip_name.replace(".zip", "").replace(".ZIP", "")

    if not file_year_text.isdigit():
        continue

    folder_year = int(folder_year_text)
    file_year = int(file_year_text)

    if folder_year != file_year:
        print(f"Warning: folder year and file year differ: {file_path}")

    zip_records.append({
        "year": folder_year,
        "filename": file_path
    })

if not zip_records:
    raise FileNotFoundError(
        f"No annual ZIP files found in {RASTER_REPO_ID} "
        f"using pattern {RASTER_BASE_FOLDER}/YYYY/YYYY.zip."
    )

zip_table = (
    pd.DataFrame(zip_records)
    .sort_values("year")
    .reset_index(drop=True)
)

# ------------------------------------------------------------------------------
# Keep only the desired processing period
# ------------------------------------------------------------------------------

zip_table = zip_table[
    (zip_table["year"] >= START_YEAR) &
    (zip_table["year"] <= END_YEAR)
].copy()

if zip_table.empty:
    raise FileNotFoundError(
        f"Annual ZIP files exist, but none were found in the target period "
        f"{START_YEAR}-{END_YEAR}."
    )

annual_zip_files = zip_table["filename"].tolist()
annual_zip_years = zip_table["year"].tolist()

print("--------------------------------------------------")
print(f"Annual ZIP files found in target period: {len(annual_zip_files):,}")
print(f"First year found                     : {min(annual_zip_years)}")
print(f"Last year found                      : {max(annual_zip_years)}")
print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Check for missing years
# ------------------------------------------------------------------------------

expected_years = list(range(START_YEAR, END_YEAR + 1))
found_years = sorted(annual_zip_years)

missing_years = sorted(set(expected_years) - set(found_years))
extra_years = sorted(set(found_years) - set(expected_years))

if missing_years:
    print("Warning: missing expected annual ZIP years:")
    print(missing_years)
else:
    print(f"Good: all expected years {START_YEAR}-{END_YEAR} were found.")

if extra_years:
    print("Extra years found outside expected period:")
    print(extra_years)

# ------------------------------------------------------------------------------
# Expected Bolivia panel size
# ------------------------------------------------------------------------------

if "EXPECTED_UNITS" in globals():
    expected_months = 0

    for year in expected_years:
        first_month = FIRST_AVAILABLE_MONTH_BY_YEAR.get(year, 1)
        expected_months += 12 - first_month + 1

    expected_rows = EXPECTED_UNITS * expected_months

    print("--------------------------------------------------")
    print("Expected Bolivia output size")
    print(f"Municipalities : {EXPECTED_UNITS:,}")
    print(f"Months         : {expected_months:,}")
    print(f"Rows           : {expected_rows:,}")
    print("--------------------------------------------------")

print("First annual ZIP files:")
display(zip_table.head())

print("Last annual ZIP files:")
display(zip_table.tail())

### Smoke test: one year only

In [ ]:
# ------------------------------------------------------------------------------
# Cell 9 — Smoke test for one annual HKU VIIRS ZIP
# ------------------------------------------------------------------------------
# Purpose:
# 1. Confirm that the public raster ZIP can be downloaded without a token.
# 2. Inspect the ZIP contents.
# 3. Identify monthly GeoTIFF files inside the ZIP.
# 4. Extract and inspect one sample monthly raster.
# 5. Compare raster bounds with the Bolivia municipal polygons.
#
# This cell does NOT calculate zonal statistics yet.

SMOKE_TEST_YEAR = 1992

print(f"Running smoke test for {COUNTRY_NAME}, year: {SMOKE_TEST_YEAR}")

# ------------------------------------------------------------------------------
# Confirm that previous cells have been run
# ------------------------------------------------------------------------------

required_objects = [
    "annual_zip_files",
    "YEAR_ZIP_TEMPLATE",
    "TEMP_DIR",
    "RASTER_REPO_ID",
    "RASTER_REPO_TYPE",
    "RASTER_TOKEN",
    "gdf_admin",
    "ID_COL",
    "FIRST_AVAILABLE_MONTH_BY_YEAR"
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise RuntimeError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_objects)
        + ". Run the configuration, vector-loading, and Cell 8 before this smoke test."
    )

if COUNTRY_NAME.lower() != "bolivia":
    print(
        f"Warning: COUNTRY_NAME is currently '{COUNTRY_NAME}'. "
        "This smoke test is configured for the Bolivia case."
    )

if ID_COL not in gdf_admin.columns:
    raise ValueError(
        f"ID column '{ID_COL}' was not found in gdf_admin. "
        "Run the Bolivia feature-standardization cell before this smoke test."
    )

print("--------------------------------------------------")
print("Bolivia vector status")
print("--------------------------------------------------")
print(f"Features        : {len(gdf_admin):,}")
print(f"Unique ID_ADMIN : {gdf_admin[ID_COL].nunique():,}")

if "EXPECTED_UNITS" in globals():
    print(f"Expected units  : {EXPECTED_UNITS:,}")

print(f"Vector CRS      : {gdf_admin.crs}")
print(f"Vector bounds   : {gdf_admin.total_bounds}")
print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Confirm the annual ZIP exists in the file list from Cell 8
# ------------------------------------------------------------------------------

smoke_zip_filename = YEAR_ZIP_TEMPLATE.format(year=SMOKE_TEST_YEAR)

if smoke_zip_filename not in annual_zip_files:
    raise FileNotFoundError(
        f"Smoke-test ZIP was not found in annual_zip_files: {smoke_zip_filename}\n"
        "Check the repository structure printed in Cell 8."
    )

print(f"ZIP file confirmed in repository list: {smoke_zip_filename}")

# ------------------------------------------------------------------------------
# Download annual ZIP from public raster repository
# ------------------------------------------------------------------------------

smoke_dir = TEMP_DIR / f"smoke_test_{SMOKE_TEST_YEAR}"
smoke_dir.mkdir(parents=True, exist_ok=True)

smoke_zip_path = hf_hub_download(
    repo_id=RASTER_REPO_ID,
    filename=smoke_zip_filename,
    repo_type=RASTER_REPO_TYPE,
    token=RASTER_TOKEN,
    local_dir=str(smoke_dir)
)

smoke_zip_path = Path(smoke_zip_path)

print("--------------------------------------------------")
print(f"Downloaded ZIP : {smoke_zip_path}")
print(f"ZIP size       : {smoke_zip_path.stat().st_size / (1024 ** 3):.2f} GB")
print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Inspect ZIP contents
# ------------------------------------------------------------------------------

with zipfile.ZipFile(smoke_zip_path, "r") as zf:
    zip_members = zf.namelist()

print(f"Files/folders inside ZIP: {len(zip_members):,}")
print("First 30 ZIP members:")
for member in zip_members[:30]:
    print(" -", member)

monthly_raster_members = [
    m for m in zip_members
    if m.lower().endswith((".tif", ".tiff")) and not m.endswith("/")
]

if not monthly_raster_members:
    raise FileNotFoundError(
        "No GeoTIFF files were found inside the ZIP. "
        "Inspect the printed ZIP members and adjust the raster filter."
    )

print("--------------------------------------------------")
print(f"Monthly raster candidates found: {len(monthly_raster_members):,}")
print("Monthly raster candidates:")
for member in monthly_raster_members:
    print(" -", member)
print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Parse year and month from raster filenames
# ------------------------------------------------------------------------------

def parse_year_month_from_name(name):
    """
    Extract year and month from a filename/path.

    Supported examples:
    - 199204.tif
    - 1992_04.tif
    - 1992-04.tif
    - something_1992_04.tif
    - something_199204.tif
    - folder/1992/04.tif
    """
    text = str(name).replace("/", "_")
    base = Path(name).name

    patterns = [
        r"(?P<year>19\d{2}|20\d{2})[_-]?(?P<month>0[1-9]|1[0-2])",
        r"(?P<year>19\d{2}|20\d{2}).*?month[_-]?(?P<month>0[1-9]|1[0-2])",
        r"(?P<year>19\d{2}|20\d{2}).*?m[_-]?(?P<month>0[1-9]|1[0-2])",
        r"(?P<year>19\d{2}|20\d{2})[_-](?P<month>0[1-9]|1[0-2])"
    ]

    for target in [base, text]:
        for pattern in patterns:
            match = re.search(pattern, target, flags=re.IGNORECASE)
            if match:
                return int(match.group("year")), int(match.group("month"))

    return None, None


parsed_months = []
unparsed_members = []

for member in monthly_raster_members:
    year, month = parse_year_month_from_name(member)

    if year is None or month is None:
        unparsed_members.append(member)
    else:
        parsed_months.append({
            "member": member,
            "year": year,
            "month": month
        })

parsed_months_df = pd.DataFrame(
    parsed_months,
    columns=["member", "year", "month"]
)

if not parsed_months_df.empty:
    parsed_months_df = (
        parsed_months_df
        .sort_values(["year", "month", "member"])
        .reset_index(drop=True)
    )

print("Parsed monthly rasters:")
display(parsed_months_df)

if unparsed_members:
    print("Warning: some raster names could not be parsed:")
    for member in unparsed_members:
        print(" -", member)

# ------------------------------------------------------------------------------
# Validate expected months
# ------------------------------------------------------------------------------

expected_smoke_months = list(
    range(FIRST_AVAILABLE_MONTH_BY_YEAR.get(SMOKE_TEST_YEAR, 1), 13)
)

if not parsed_months_df.empty:
    found_smoke_months = sorted(
        parsed_months_df.loc[
            parsed_months_df["year"] == SMOKE_TEST_YEAR,
            "month"
        ].unique().tolist()
    )
else:
    found_smoke_months = []

missing_smoke_months = sorted(
    set(expected_smoke_months) - set(found_smoke_months)
)

print("--------------------------------------------------")
print(f"Expected months for {SMOKE_TEST_YEAR}: {expected_smoke_months}")
print(f"Found months for {SMOKE_TEST_YEAR}    : {found_smoke_months}")
print(f"Missing months                      : {missing_smoke_months}")
print("--------------------------------------------------")

if missing_smoke_months:
    print("Warning: some expected months are missing from the smoke-test ZIP.")

if "EXPECTED_UNITS" in globals():
    expected_smoke_rows = EXPECTED_UNITS * len(expected_smoke_months)

    print("--------------------------------------------------")
    print("Expected Bolivia smoke-test zonal-stat size")
    print(f"Municipalities : {EXPECTED_UNITS:,}")
    print(f"Months         : {len(expected_smoke_months):,}")
    print(f"Expected rows  : {expected_smoke_rows:,}")
    print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Extract and inspect one sample raster
# ------------------------------------------------------------------------------

if parsed_months_df.empty:
    sample_member = monthly_raster_members[0]
    sample_year = None
    sample_month = None
else:
    sample_row = parsed_months_df.iloc[0]
    sample_member = sample_row["member"]
    sample_year = int(sample_row["year"])
    sample_month = int(sample_row["month"])

sample_extract_dir = smoke_dir / "sample_extract"

if sample_extract_dir.exists():
    shutil.rmtree(sample_extract_dir, ignore_errors=True)

sample_extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(smoke_zip_path, "r") as zf:
    zf.extract(sample_member, path=sample_extract_dir)

sample_raster_path = sample_extract_dir / sample_member

print(f"Sample raster extracted: {sample_raster_path}")
print(f"Parsed sample year/month: {sample_year}-{sample_month}")

with rasterio.open(sample_raster_path) as src:
    sample_raster_crs = src.crs
    sample_raster_bounds = src.bounds

    print("--------------------------------------------------")
    print("Sample raster metadata")
    print("--------------------------------------------------")
    print(f"CRS          : {src.crs}")
    print(f"Nodata       : {src.nodata}")
    print(f"Shape        : {src.height:,} rows x {src.width:,} cols")
    print(f"Bounds       : {src.bounds}")
    print(f"Resolution   : {src.res}")
    print(f"Dtype        : {src.dtypes}")
    print(f"Count/bands  : {src.count}")

# ------------------------------------------------------------------------------
# Compare Bolivia vector bounds with raster bounds
# ------------------------------------------------------------------------------

print("--------------------------------------------------")
print("Vector-raster spatial check")
print("--------------------------------------------------")
print(f"{COUNTRY_NAME} vector CRS     : {gdf_admin.crs}")
print(f"Sample raster CRS             : {sample_raster_crs}")
print(f"{COUNTRY_NAME} vector bounds  : {gdf_admin.total_bounds}")
print(f"Sample raster bounds          : {sample_raster_bounds}")
print("--------------------------------------------------")

if sample_raster_crs is not None and gdf_admin.crs is not None:
    try:
        gdf_for_bounds = gdf_admin.to_crs(sample_raster_crs)
        print(f"{COUNTRY_NAME} bounds in raster CRS: {gdf_for_bounds.total_bounds}")
    except Exception as e:
        print("Warning: could not reproject vector bounds to raster CRS.")
        print(e)

print("Smoke test completed successfully.")

### Monthly statisctics for smoke test

In [ ]:
# ------------------------------------------------------------------------------
# Cell 10 — Calculate monthly zonal statistics for one year
# ------------------------------------------------------------------------------
# This cell calculates zonal statistics for all monthly rasters in one annual ZIP.
#
# For Bolivia 1992:
# - expected months: April-December
# - expected rows: number of Bolivian municipalities × 9 months
#
# Output objects:
# - df_year_stats
# - YEAR_CHECKPOINT_PARQUET
# - YEAR_CHECKPOINT_CSV

TARGET_YEAR = 1992

print(f"Calculating monthly zonal statistics for {COUNTRY_NAME}, year: {TARGET_YEAR}")

# ------------------------------------------------------------------------------
# Options for this one-year run
# ------------------------------------------------------------------------------

SAVE_YEAR_CHECKPOINT = True
CLEAN_EXTRACTED_RASTERS_AFTER_PROCESS = True
DELETE_YEAR_ZIP_AFTER_PROCESS = False  # keep False while testing/debugging

# ------------------------------------------------------------------------------
# Required objects check
# ------------------------------------------------------------------------------

required_objects = [
    "gdf_admin",
    "ATTRIBUTE_COLS",
    "geometry_col",
    "annual_zip_files",
    "YEAR_ZIP_TEMPLATE",
    "RASTER_REPO_ID",
    "RASTER_REPO_TYPE",
    "RASTER_TOKEN",
    "MODE_SETTINGS",
    "PROCESSING_MODE",
    "ZONAL_STATS",
    "CUSTOM_ZONAL_STATS",
    "ALL_ZONAL_OUTPUT_STATS",
    "ALL_TOUCHED",
    "CHECKPOINT_DIR",
    "TEMP_DIR",
    "COUNTRY_NAME",
    "PRODUCT_NAME",
    "PROJECT_NAME",
    "FIRST_AVAILABLE_MONTH_BY_YEAR",
    "ID_COL",
    "total_cores",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise RuntimeError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_objects)
    )

if COUNTRY_NAME.lower() != "bolivia":
    print(
        f"Warning: COUNTRY_NAME is currently '{COUNTRY_NAME}'. "
        "This one-year run is configured for the Bolivia municipal case."
    )

if ID_COL not in gdf_admin.columns:
    raise ValueError(
        f"ID column '{ID_COL}' was not found in gdf_admin. "
        "Run the Bolivia feature-standardization cell before this cell."
    )

if "EXPECTED_UNITS" in globals():
    print("--------------------------------------------------")
    print("Bolivia vector status")
    print(f"Municipalities loaded : {len(gdf_admin):,}")
    print(f"Unique ID_ADMIN       : {gdf_admin[ID_COL].nunique():,}")
    print(f"Expected units        : {EXPECTED_UNITS:,}")
    print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Locate or download the annual ZIP
# ------------------------------------------------------------------------------

year_zip_filename = YEAR_ZIP_TEMPLATE.format(year=TARGET_YEAR)

# If the smoke test already downloaded the ZIP, reuse it.
if "smoke_zip_path" in globals() and Path(smoke_zip_path).exists():
    year_zip_path = Path(smoke_zip_path)
    print(f"Using ZIP already downloaded by smoke test: {year_zip_path}")
else:
    year_download_dir = TEMP_DIR / f"year_download_{TARGET_YEAR}"
    year_download_dir.mkdir(parents=True, exist_ok=True)

    year_zip_path = hf_hub_download(
        repo_id=RASTER_REPO_ID,
        filename=year_zip_filename,
        repo_type=RASTER_REPO_TYPE,
        token=RASTER_TOKEN,
        local_dir=str(year_download_dir)
    )

    year_zip_path = Path(year_zip_path)
    print(f"Downloaded ZIP: {year_zip_path}")

print("--------------------------------------------------")
print(f"ZIP path : {year_zip_path}")
print(f"ZIP size : {year_zip_path.stat().st_size / (1024 ** 3):.2f} GB")
print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Helper: parse year and month from raster filename
# ------------------------------------------------------------------------------

def parse_year_month_from_name(name):
    """
    Extract year and month from a filename/path.

    Supported examples:
    - 199204.tif
    - 1992_04.tif
    - 1992-04.tif
    - SVDNB_npp_simulated_199204.tif
    - folder/1992/SVDNB_npp_simulated_199204.tif
    """
    text = str(name).replace("/", "_")
    base = Path(name).name

    patterns = [
        r"(?P<year>19\d{2}|20\d{2})[_-]?(?P<month>0[1-9]|1[0-2])",
        r"(?P<year>19\d{2}|20\d{2}).*?month[_-]?(?P<month>0[1-9]|1[0-2])",
        r"(?P<year>19\d{2}|20\d{2}).*?m[_-]?(?P<month>0[1-9]|1[0-2])",
    ]

    for target in [base, text]:
        for pattern in patterns:
            match = re.search(pattern, target, flags=re.IGNORECASE)
            if match:
                return int(match.group("year")), int(match.group("month"))

    return None, None

# ------------------------------------------------------------------------------
# Read ZIP members and identify monthly rasters for TARGET_YEAR
# ------------------------------------------------------------------------------

with zipfile.ZipFile(year_zip_path, "r") as zf:
    zip_members = zf.namelist()

monthly_raster_members = [
    m for m in zip_members
    if m.lower().endswith((".tif", ".tiff")) and not m.endswith("/")
]

if not monthly_raster_members:
    raise FileNotFoundError(
        f"No GeoTIFF files found inside ZIP: {year_zip_path}"
    )

monthly_records = []

for member in monthly_raster_members:
    year, month = parse_year_month_from_name(member)

    if year == TARGET_YEAR:
        monthly_records.append({
            "member": member,
            "year": year,
            "month": month
        })

year_months_df = (
    pd.DataFrame(monthly_records)
    .sort_values(["year", "month", "member"])
    .reset_index(drop=True)
)

if year_months_df.empty:
    raise RuntimeError(
        f"No monthly rasters for TARGET_YEAR={TARGET_YEAR} were parsed from {year_zip_path}."
    )

print("Monthly rasters selected for processing:")
display(year_months_df)

# Validate expected months
expected_months = list(range(FIRST_AVAILABLE_MONTH_BY_YEAR.get(TARGET_YEAR, 1), 13))
found_months = sorted(year_months_df["month"].unique().tolist())
missing_months = sorted(set(expected_months) - set(found_months))

print("--------------------------------------------------")
print(f"Expected months : {expected_months}")
print(f"Found months    : {found_months}")
print(f"Missing months  : {missing_months}")
print("--------------------------------------------------")

if missing_months:
    raise RuntimeError(
        f"Missing monthly rasters for {TARGET_YEAR}: {missing_months}"
    )

# ------------------------------------------------------------------------------
# Extract all monthly rasters for this year
# ------------------------------------------------------------------------------

year_extract_dir = TEMP_DIR / f"extracted_{TARGET_YEAR}"

if year_extract_dir.exists():
    shutil.rmtree(year_extract_dir, ignore_errors=True)

year_extract_dir.mkdir(parents=True, exist_ok=True)

print(f"Extracting {len(year_months_df)} monthly rasters to: {year_extract_dir}")

with zipfile.ZipFile(year_zip_path, "r") as zf:
    for member in year_months_df["member"].tolist():
        zf.extract(member, path=year_extract_dir)

year_months_df["raster_path"] = year_months_df["member"].apply(
    lambda m: str(year_extract_dir / m)
)

# Check that all extracted rasters exist
missing_extracted = [
    p for p in year_months_df["raster_path"].tolist()
    if not Path(p).exists()
]

if missing_extracted:
    raise FileNotFoundError(
        "Some monthly rasters were not extracted correctly:\n"
        + "\n".join(missing_extracted[:10])
    )

print("Extraction completed.")

# ------------------------------------------------------------------------------
# Inspect raster CRS and prepare polygons for statistics
# ------------------------------------------------------------------------------

sample_raster_path = year_months_df.loc[0, "raster_path"]

with rasterio.open(sample_raster_path) as src:
    raster_crs = src.crs
    raster_nodata = src.nodata
    raster_bounds = src.bounds
    raster_res = src.res
    raster_shape = (src.height, src.width)
    raster_dtype = src.dtypes[0]

print("--------------------------------------------------")
print("Sample raster metadata for zonal statistics")
print("--------------------------------------------------")
print(f"Sample raster : {sample_raster_path}")
print(f"Raster CRS    : {raster_crs}")
print(f"Raster nodata : {raster_nodata}")
print(f"Raster bounds : {raster_bounds}")
print(f"Raster res    : {raster_res}")
print(f"Raster shape  : {raster_shape[0]:,} rows x {raster_shape[1]:,} cols")
print(f"Raster dtype  : {raster_dtype}")
print("--------------------------------------------------")

if raster_crs is None:
    raise ValueError(
        "Raster CRS is None. Stop here and define a manual CRS strategy before computing zonal statistics."
    )

if gdf_admin.crs != raster_crs:
    print(f"Reprojecting Bolivia municipal polygons from {gdf_admin.crs} to {raster_crs}...")
    gdf_for_stats_year = (
        gdf_admin[ATTRIBUTE_COLS + [geometry_col]]
        .to_crs(raster_crs)
        .copy()
    )
else:
    print("Vector CRS already matches raster CRS. No reprojection needed.")
    gdf_for_stats_year = gdf_admin[ATTRIBUTE_COLS + [geometry_col]].copy()

# Make sure the geometry column is active
gdf_for_stats_year = gdf_for_stats_year.set_geometry(geometry_col)

print(f"Bolivia municipal polygons ready for zonal statistics: {len(gdf_for_stats_year):,}")

# ------------------------------------------------------------------------------
# Choose monthly workers
# ------------------------------------------------------------------------------

def choose_month_workers(number_of_monthly_rasters):
    """
    Choose a safe number of parallel workers for month-level processing.
    """
    settings = MODE_SETTINGS[PROCESSING_MODE]

    current_available_ram_gb = psutil.virtual_memory().available / (1024 ** 3)

    workers = min(
        number_of_monthly_rasters,
        max(1, total_cores - 2),
        max(1, int(current_available_ram_gb / settings["ram_per_worker_gb"])),
        settings["max_month_workers"]
    )

    return max(1, workers)


month_workers = choose_month_workers(len(year_months_df))

print("--------------------------------------------------")
print(f"Processing mode          : {PROCESSING_MODE}")
print(f"Monthly rasters          : {len(year_months_df)}")
print(f"CPU logical cores        : {total_cores}")
print(f"Available RAM now        : {psutil.virtual_memory().available / (1024 ** 3):.2f} GB")
print(f"Monthly workers selected : {month_workers}")
print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Worker function: process one monthly raster
# ------------------------------------------------------------------------------

def process_single_month(row):
    """
    Calculate zonal statistics for one monthly raster.
    """
    member = row["member"]
    year = int(row["year"])
    month = int(row["month"])
    raster_path = row["raster_path"]

    try:
        with rasterio.open(raster_path) as src:
            month_nodata = src.nodata

        stats = zonal_stats(
            vectors=gdf_for_stats_year.geometry,
            raster=raster_path,
            stats=ZONAL_STATS,
            add_stats=CUSTOM_ZONAL_STATS,
            nodata=month_nodata,
            all_touched=ALL_TOUCHED,
            geojson_out=False
        )

        df_stats = pd.DataFrame(stats)
        df_attrs = gdf_for_stats_year[ATTRIBUTE_COLS].reset_index(drop=True).copy()

        df_month = pd.concat([df_attrs, df_stats], axis=1)

        df_month["country"] = COUNTRY_NAME
        df_month["year"] = year
        df_month["month"] = month
        df_month["product"] = PRODUCT_NAME
        df_month["source_member"] = member
        df_month["source_zip"] = year_zip_filename

        print(f"Finished {year}-{month:02d}: {Path(member).name}")

        return df_month

    except Exception as e:
        print(f"Failed {year}-{month:02d}: {member}")
        print(e)
        traceback.print_exc()
        return None

# ------------------------------------------------------------------------------
# Run monthly zonal statistics in parallel
# ------------------------------------------------------------------------------

print(f"Starting zonal statistics for {COUNTRY_NAME}, {TARGET_YEAR}...")

monthly_rows = year_months_df.to_dict("records")

monthly_results = Parallel(
    n_jobs=month_workers,
    backend="loky",
    verbose=5
)(
    delayed(process_single_month)(row)
    for row in monthly_rows
)

valid_monthly_results = [
    df for df in monthly_results
    if df is not None
]

if not valid_monthly_results:
    raise RuntimeError(f"No valid monthly results were generated for {TARGET_YEAR}.")

df_year_stats = pd.concat(valid_monthly_results, ignore_index=True)

# ------------------------------------------------------------------------------
# Clean, order, and validate output
# ------------------------------------------------------------------------------

stat_cols = [c for c in ALL_ZONAL_OUTPUT_STATS if c in df_year_stats.columns]

preferred_cols = (
    [ID_COL]
    + [c for c in ATTRIBUTE_COLS if c != ID_COL and c in df_year_stats.columns]
    + ["country", "year", "month", "product"]
    + stat_cols
    + ["source_member", "source_zip"]
)

remaining_cols = [
    c for c in df_year_stats.columns
    if c not in preferred_cols
]

df_year_stats = df_year_stats[preferred_cols + remaining_cols]

df_year_stats = df_year_stats.sort_values(
    ["year", "month", ID_COL]
).reset_index(drop=True)

expected_rows = len(gdf_admin) * len(year_months_df)

print("--------------------------------------------------")
print(f"Completed year       : {TARGET_YEAR}")
print(f"Months processed     : {len(year_months_df)}")
print(f"Municipalities       : {len(gdf_admin):,}")
print(f"Expected rows        : {expected_rows:,}")
print(f"Actual rows          : {len(df_year_stats):,}")
print("--------------------------------------------------")

if len(df_year_stats) != expected_rows:
    print("Warning: row count does not match expected rows.")
else:
    print("Good: row count matches municipalities × months.")

if "EXPECTED_UNITS" in globals():
    expected_rows_from_config = EXPECTED_UNITS * len(year_months_df)

    print("--------------------------------------------------")
    print("Configured Bolivia expectation")
    print(f"Expected units from config : {EXPECTED_UNITS:,}")
    print(f"Expected rows from config  : {expected_rows_from_config:,}")
    print("--------------------------------------------------")

    if len(df_year_stats) != expected_rows_from_config:
        print("Warning: row count does not match EXPECTED_UNITS × months.")
    else:
        print("Good: row count matches EXPECTED_UNITS × months.")

print("Rows by year-month:")
display(
    df_year_stats
    .groupby(["year", "month"])
    .size()
    .reset_index(name="rows")
)

print("Missing values in main statistics:")
display(
    df_year_stats[stat_cols]
    .isna()
    .sum()
    .reset_index(name="missing_values")
    .rename(columns={"index": "variable"})
)

# ------------------------------------------------------------------------------
# Diagnostic checks for count, zeros, positive, and invalid pixels
# ------------------------------------------------------------------------------

diagnostic_cols = [
    c for c in ["count", "count_zeros", "count_positive", "count_invalid"]
    if c in df_year_stats.columns
]

if diagnostic_cols:
    print("Pixel-count diagnostics:")
    display(
        df_year_stats[diagnostic_cols]
        .describe()
        .T
    )

if all(c in df_year_stats.columns for c in ["count", "count_zeros", "count_positive"]):
    df_year_stats["count_zero_positive_gap"] = (
        df_year_stats["count"]
        - df_year_stats["count_zeros"]
        - df_year_stats["count_positive"]
    )

    print("Check: count - count_zeros - count_positive")
    display(
        df_year_stats["count_zero_positive_gap"]
        .describe()
        .to_frame(name="count_zero_positive_gap")
    )

    nonzero_gap_rows = int((df_year_stats["count_zero_positive_gap"] != 0).sum())

    if nonzero_gap_rows == 0:
        print("Good: count equals count_zeros + count_positive for all rows.")
    else:
        print(
            f"Warning: {nonzero_gap_rows:,} rows have count != count_zeros + count_positive. "
            "This may indicate negative valid raster values or another valid value category."
        )

print("Preview:")
display(df_year_stats.head())

# ------------------------------------------------------------------------------
# Save yearly checkpoint
# ------------------------------------------------------------------------------

YEAR_CHECKPOINT_PARQUET = CHECKPOINT_DIR / f"{PROJECT_NAME}_{TARGET_YEAR}.parquet"
YEAR_CHECKPOINT_CSV = CHECKPOINT_DIR / f"{PROJECT_NAME}_{TARGET_YEAR}.csv"

if SAVE_YEAR_CHECKPOINT:
    df_year_stats.to_parquet(YEAR_CHECKPOINT_PARQUET, index=False)
    df_year_stats.to_csv(YEAR_CHECKPOINT_CSV, index=False)

    print("--------------------------------------------------")
    print("Yearly checkpoint saved:")
    print(f"Parquet : {YEAR_CHECKPOINT_PARQUET}")
    print(f"CSV     : {YEAR_CHECKPOINT_CSV}")
    print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# Clean temporary extracted rasters
# ------------------------------------------------------------------------------

if CLEAN_EXTRACTED_RASTERS_AFTER_PROCESS:
    if year_extract_dir.exists():
        shutil.rmtree(year_extract_dir, ignore_errors=True)
        print(f"Deleted extracted raster directory: {year_extract_dir}")

if DELETE_YEAR_ZIP_AFTER_PROCESS:
    if year_zip_path.exists():
        year_zip_path.unlink()
        print(f"Deleted annual ZIP: {year_zip_path}")

gc.collect()

print(f"Cell 10 completed successfully for {COUNTRY_NAME}, {TARGET_YEAR}.")

### Spaghetti plot for smoke test

In [ ]:
# ==============================================================================
# SPAGHETTI PLOT: MONTHLY MUNICIPALITY-LEVEL SUM, 1992
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------------------------

YEAR_TO_PLOT = 1992
VALUE_COLUMN = "sum"

CHECKPOINT_PARQUET = CHECKPOINT_DIR / f"{PROJECT_NAME}_{YEAR_TO_PLOT}.parquet"
CHECKPOINT_CSV = CHECKPOINT_DIR / f"{PROJECT_NAME}_{YEAR_TO_PLOT}.csv"

FIGURE_DIR = BASE_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FIGURE = FIGURE_DIR / f"spaghetti_plot_municipality_sum_{YEAR_TO_PLOT}.png"

# Municipality identifier and optional name column for Bolivia.
MUNICIPALITY_ID_COLUMN = ID_COL

# Try to use the best available municipality-name column.
municipality_name_candidates = [
    "mun_name",
    "municipality_name",
    "municipio",
    "name",
    "NAME"
]

MUNICIPALITY_NAME_COLUMN = next(
    (col for col in municipality_name_candidates if col in globals().get("ATTRIBUTE_COLS", [])),
    None
)

# Optional transformation.
# Use False for raw sum.
# Use True if the raw lines are too dominated by very bright municipalities.
USE_LOG1P = False

# ------------------------------------------------------------------------------
# 2. Load yearly checkpoint
# ------------------------------------------------------------------------------

if CHECKPOINT_PARQUET.exists():
    df = pd.read_parquet(CHECKPOINT_PARQUET)
    print(f"Loaded Parquet checkpoint: {CHECKPOINT_PARQUET}")
elif CHECKPOINT_CSV.exists():
    df = pd.read_csv(CHECKPOINT_CSV)
    print(f"Loaded CSV checkpoint: {CHECKPOINT_CSV}")
else:
    raise FileNotFoundError(
        f"No checkpoint found for {YEAR_TO_PLOT}. "
        f"Expected either {CHECKPOINT_PARQUET} or {CHECKPOINT_CSV}."
    )

print("Data shape:", df.shape)
print("Columns:", list(df.columns))

# ------------------------------------------------------------------------------
# 3. Validate required columns
# ------------------------------------------------------------------------------

required_columns = ["year", "month", VALUE_COLUMN, MUNICIPALITY_ID_COLUMN]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Keep only the selected year.
plot_df = df[df["year"] == YEAR_TO_PLOT].copy()

if plot_df.empty:
    raise ValueError(f"No rows found for year {YEAR_TO_PLOT}.")

# Make sure month and value are numeric.
plot_df["month"] = pd.to_numeric(plot_df["month"], errors="coerce")
plot_df[VALUE_COLUMN] = pd.to_numeric(plot_df[VALUE_COLUMN], errors="coerce")

plot_df = plot_df.dropna(
    subset=["month", VALUE_COLUMN, MUNICIPALITY_ID_COLUMN]
)

# ------------------------------------------------------------------------------
# 4. Build municipality-month matrix
# ------------------------------------------------------------------------------

municipality_month = (
    plot_df
    .pivot_table(
        index=MUNICIPALITY_ID_COLUMN,
        columns="month",
        values=VALUE_COLUMN,
        aggfunc="first"
    )
    .sort_index(axis=1)
)

print("\nMunicipality-month matrix")
print("-" * 70)
print("Municipalities:", municipality_month.shape[0])
print("Months        :", list(municipality_month.columns))
print("Shape         :", municipality_month.shape)

if "EXPECTED_UNITS" in globals():
    print("Expected municipalities:", EXPECTED_UNITS)

# Optional log transformation.
if USE_LOG1P:
    municipality_month_to_plot = municipality_month.apply(lambda x: np.log1p(x))
    y_label = f"log(1 + {VALUE_COLUMN})"
else:
    municipality_month_to_plot = municipality_month
    y_label = VALUE_COLUMN

# Monthly aggregate lines for reference.
monthly_median = municipality_month_to_plot.median(axis=0)
monthly_mean = municipality_month_to_plot.mean(axis=0)

# ------------------------------------------------------------------------------
# 5. Plot
# ------------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 7))

# One line per municipality.
for municipality_id, row in municipality_month_to_plot.iterrows():
    ax.plot(
        row.index,
        row.values,
        linewidth=0.8,
        alpha=0.18
    )

# Add median and mean as reference lines.
ax.plot(
    monthly_median.index,
    monthly_median.values,
    linewidth=2.8,
    label="Monthly median"
)

ax.plot(
    monthly_mean.index,
    monthly_mean.values,
    linewidth=2.8,
    linestyle="--",
    label="Monthly mean"
)

ax.set_title(
    f"Bolivia municipalities: monthly nighttime lights sum, {YEAR_TO_PLOT}",
    fontsize=15,
    pad=14
)

ax.set_xlabel("Month")
ax.set_ylabel(y_label)

ax.set_xticks(sorted(municipality_month_to_plot.columns))
ax.grid(True, alpha=0.25)
ax.legend()

plt.tight_layout()

plt.savefig(OUTPUT_FIGURE, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figure saved to: {OUTPUT_FIGURE}")

In [ ]:
# ==============================================================================
# SPAGHETTI PLOT: MONTHLY MUNICIPALITY-LEVEL SUM, 1992
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------------------------

YEAR_TO_PLOT = 1992
VALUE_COLUMN = "sum"

CHECKPOINT_PARQUET = CHECKPOINT_DIR / f"{PROJECT_NAME}_{YEAR_TO_PLOT}.parquet"
CHECKPOINT_CSV = CHECKPOINT_DIR / f"{PROJECT_NAME}_{YEAR_TO_PLOT}.csv"

FIGURE_DIR = BASE_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FIGURE = FIGURE_DIR / f"spaghetti_plot_municipality_sum_{YEAR_TO_PLOT}.png"

# Municipality identifier and optional name column for Bolivia.
MUNICIPALITY_ID_COLUMN = ID_COL

# Try to use the best available municipality-name column.
municipality_name_candidates = [
    "mun_name",
    "municipality_name",
    "municipio",
    "name",
    "NAME"
]

MUNICIPALITY_NAME_COLUMN = next(
    (col for col in municipality_name_candidates if col in globals().get("ATTRIBUTE_COLS", [])),
    None
)

# Optional transformation.
# Use False for raw sum.
# Use True if the raw lines are too dominated by very bright municipalities.
USE_LOG1P = True

# ------------------------------------------------------------------------------
# 2. Load yearly checkpoint
# ------------------------------------------------------------------------------

if CHECKPOINT_PARQUET.exists():
    df = pd.read_parquet(CHECKPOINT_PARQUET)
    print(f"Loaded Parquet checkpoint: {CHECKPOINT_PARQUET}")
elif CHECKPOINT_CSV.exists():
    df = pd.read_csv(CHECKPOINT_CSV)
    print(f"Loaded CSV checkpoint: {CHECKPOINT_CSV}")
else:
    raise FileNotFoundError(
        f"No checkpoint found for {YEAR_TO_PLOT}. "
        f"Expected either {CHECKPOINT_PARQUET} or {CHECKPOINT_CSV}."
    )

print("Data shape:", df.shape)
print("Columns:", list(df.columns))

# ------------------------------------------------------------------------------
# 3. Validate required columns
# ------------------------------------------------------------------------------

required_columns = ["year", "month", VALUE_COLUMN, MUNICIPALITY_ID_COLUMN]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Keep only the selected year.
plot_df = df[df["year"] == YEAR_TO_PLOT].copy()

if plot_df.empty:
    raise ValueError(f"No rows found for year {YEAR_TO_PLOT}.")

# Make sure month and value are numeric.
plot_df["month"] = pd.to_numeric(plot_df["month"], errors="coerce")
plot_df[VALUE_COLUMN] = pd.to_numeric(plot_df[VALUE_COLUMN], errors="coerce")

plot_df = plot_df.dropna(
    subset=["month", VALUE_COLUMN, MUNICIPALITY_ID_COLUMN]
)

# ------------------------------------------------------------------------------
# 4. Build municipality-month matrix
# ------------------------------------------------------------------------------

municipality_month = (
    plot_df
    .pivot_table(
        index=MUNICIPALITY_ID_COLUMN,
        columns="month",
        values=VALUE_COLUMN,
        aggfunc="first"
    )
    .sort_index(axis=1)
)

print("\nMunicipality-month matrix")
print("-" * 70)
print("Municipalities:", municipality_month.shape[0])
print("Months        :", list(municipality_month.columns))
print("Shape         :", municipality_month.shape)

if "EXPECTED_UNITS" in globals():
    print("Expected municipalities:", EXPECTED_UNITS)

# Optional log transformation.
if USE_LOG1P:
    municipality_month_to_plot = municipality_month.apply(lambda x: np.log1p(x))
    y_label = f"log(1 + {VALUE_COLUMN})"
else:
    municipality_month_to_plot = municipality_month
    y_label = VALUE_COLUMN

# Monthly aggregate lines for reference.
monthly_median = municipality_month_to_plot.median(axis=0)
monthly_mean = municipality_month_to_plot.mean(axis=0)

# ------------------------------------------------------------------------------
# 5. Plot
# ------------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 7))

# One line per municipality.
for municipality_id, row in municipality_month_to_plot.iterrows():
    ax.plot(
        row.index,
        row.values,
        linewidth=0.8,
        alpha=0.18
    )

# Add median and mean as reference lines.
ax.plot(
    monthly_median.index,
    monthly_median.values,
    linewidth=2.8,
    label="Monthly median"
)

ax.plot(
    monthly_mean.index,
    monthly_mean.values,
    linewidth=2.8,
    linestyle="--",
    label="Monthly mean"
)

ax.set_title(
    f"Bolivia municipalities: monthly nighttime lights sum, {YEAR_TO_PLOT}",
    fontsize=15,
    pad=14
)

ax.set_xlabel("Month")
ax.set_ylabel(y_label)

ax.set_xticks(sorted(municipality_month_to_plot.columns))
ax.grid(True, alpha=0.25)
ax.legend()

plt.tight_layout()

plt.savefig(OUTPUT_FIGURE, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figure saved to: {OUTPUT_FIGURE}")

### Zero and NaN revision

In [ ]:
# ==============================================================================
# COUNT ZERO / POSITIVE / INVALID PIXELS BY MUNICIPALITY-YEAR
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------------------------

VALUE_COLUMN = "sum"

# Municipality identifier for Bolivia.
# This should normally be ID_ADMIN, created from asdf_id.
MUNICIPALITY_ID_COLUMN = ID_COL if "ID_COL" in globals() else "ID_ADMIN"

ZERO_OUTPUT_CSV = OUTPUT_DIR / "pixel_count_diagnostics_by_municipality_year.csv"
ZERO_OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------------------------

if "df_final" in globals():
    df_zero = df_final.copy()
    print("Using df_final from memory.")
elif "df_year_stats" in globals():
    df_zero = df_year_stats.copy()
    print("Using df_year_stats from memory.")
else:
    checkpoint_files = sorted(CHECKPOINT_DIR.glob("*.parquet"))

    if not checkpoint_files:
        raise FileNotFoundError(
            "No df_final, no df_year_stats, and no yearly Parquet checkpoints found."
        )

    print(f"Loading {len(checkpoint_files)} yearly checkpoints...")
    df_zero = pd.concat(
        [pd.read_parquet(path) for path in checkpoint_files],
        ignore_index=True
    )

print("Data shape:", df_zero.shape)

# ------------------------------------------------------------------------------
# 3. Validate required columns
# ------------------------------------------------------------------------------

required_columns = [
    MUNICIPALITY_ID_COLUMN,
    "year",
    "month",
    VALUE_COLUMN,
    "count"
]

missing_columns = [col for col in required_columns if col not in df_zero.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

optional_pixel_count_columns = [
    "count_zeros",
    "count_positive",
    "count_invalid"
]

available_pixel_count_columns = [
    col for col in optional_pixel_count_columns
    if col in df_zero.columns
]

if not available_pixel_count_columns:
    print(
        "Warning: count_zeros/count_positive/count_invalid are not available. "
        "This cell will only summarize zero months based on sum == 0."
    )

df_zero["year"] = pd.to_numeric(df_zero["year"], errors="coerce").astype("Int64")
df_zero["month"] = pd.to_numeric(df_zero["month"], errors="coerce").astype("Int64")
df_zero[VALUE_COLUMN] = pd.to_numeric(df_zero[VALUE_COLUMN], errors="coerce")
df_zero["count"] = pd.to_numeric(df_zero["count"], errors="coerce")

for col in available_pixel_count_columns:
    df_zero[col] = pd.to_numeric(df_zero[col], errors="coerce")

# ------------------------------------------------------------------------------
# 4. Build zero indicators
# ------------------------------------------------------------------------------

df_zero["value_is_zero"] = df_zero[VALUE_COLUMN].eq(0)

# Zero-light month only when the municipality has valid pixels.
df_zero["real_zero_with_valid_pixels"] = (
    df_zero[VALUE_COLUMN].eq(0) &
    df_zero["count"].fillna(0).gt(0)
)

# Optional consistency check when both pixel-count columns exist.
if all(c in df_zero.columns for c in ["count", "count_zeros", "count_positive"]):
    df_zero["count_zero_positive_gap"] = (
        df_zero["count"]
        - df_zero["count_zeros"]
        - df_zero["count_positive"]
    )
else:
    df_zero["count_zero_positive_gap"] = np.nan

# ------------------------------------------------------------------------------
# 5. Count zero / positive / invalid pixels by municipality-year
# ------------------------------------------------------------------------------

id_cols = [MUNICIPALITY_ID_COLUMN, "year"]

# Keep useful Bolivia administrative columns if available.
possible_attr_cols = [
    "asdf_id",
    "mun_id",
    "mun_name",
    "municipality_id",
    "municipality_name",
    "municipio_id",
    "municipio",
    "dep_id",
    "dep_name",
    "department_id",
    "department_name",
    "departamento",
    "name",
    "NAME"
]

# Also keep any previously selected administrative attributes.
if "ATTRIBUTE_COLS" in globals():
    possible_attr_cols = possible_attr_cols + [
        c for c in ATTRIBUTE_COLS
        if c not in possible_attr_cols and c != MUNICIPALITY_ID_COLUMN
    ]

attr_cols = [
    c for c in possible_attr_cols
    if c in df_zero.columns and c != MUNICIPALITY_ID_COLUMN
]

agg_dict = {
    "months_observed": ("month", "nunique"),
    "rows_observed": ("month", "size"),
    "zero_months": ("value_is_zero", "sum"),
    "real_zero_months": ("real_zero_with_valid_pixels", "sum"),
    "valid_pixel_months_total": ("count", "sum"),
    "mean_valid_pixels_per_month": ("count", "mean"),
    "mean_value": (VALUE_COLUMN, "mean"),
    "min_value": (VALUE_COLUMN, "min"),
    "max_value": (VALUE_COLUMN, "max"),
    "count_zero_positive_gap_sum": ("count_zero_positive_gap", "sum"),
    "count_zero_positive_gap_max_abs": (
        "count_zero_positive_gap",
        lambda x: np.nanmax(np.abs(x)) if x.notna().any() else np.nan
    )
}

if "count_zeros" in df_zero.columns:
    agg_dict.update({
        "zero_pixels_total": ("count_zeros", "sum"),
        "mean_zero_pixels_per_month": ("count_zeros", "mean"),
        "max_zero_pixels_month": ("count_zeros", "max")
    })

if "count_positive" in df_zero.columns:
    agg_dict.update({
        "positive_pixels_total": ("count_positive", "sum"),
        "mean_positive_pixels_per_month": ("count_positive", "mean"),
        "max_positive_pixels_month": ("count_positive", "max")
    })

if "count_invalid" in df_zero.columns:
    agg_dict.update({
        "invalid_pixels_total": ("count_invalid", "sum"),
        "mean_invalid_pixels_per_month": ("count_invalid", "mean"),
        "max_invalid_pixels_month": ("count_invalid", "max")
    })

pixel_counts_by_municipality_year = (
    df_zero
    .groupby(id_cols, as_index=False)
    .agg(**agg_dict)
)

pixel_counts_by_municipality_year["share_zero_months"] = (
    pixel_counts_by_municipality_year["zero_months"] /
    pixel_counts_by_municipality_year["rows_observed"]
)

pixel_counts_by_municipality_year["share_real_zero_months"] = (
    pixel_counts_by_municipality_year["real_zero_months"] /
    pixel_counts_by_municipality_year["rows_observed"]
)

if "zero_pixels_total" in pixel_counts_by_municipality_year.columns:
    pixel_counts_by_municipality_year["share_zero_pixels"] = np.where(
        pixel_counts_by_municipality_year["valid_pixel_months_total"].gt(0),
        pixel_counts_by_municipality_year["zero_pixels_total"] /
        pixel_counts_by_municipality_year["valid_pixel_months_total"],
        np.nan
    )

if "positive_pixels_total" in pixel_counts_by_municipality_year.columns:
    pixel_counts_by_municipality_year["share_positive_pixels"] = np.where(
        pixel_counts_by_municipality_year["valid_pixel_months_total"].gt(0),
        pixel_counts_by_municipality_year["positive_pixels_total"] /
        pixel_counts_by_municipality_year["valid_pixel_months_total"],
        np.nan
    )

# Add attributes back.
if attr_cols:
    attrs = (
        df_zero[[MUNICIPALITY_ID_COLUMN] + attr_cols]
        .drop_duplicates(subset=[MUNICIPALITY_ID_COLUMN])
    )

    pixel_counts_by_municipality_year = pixel_counts_by_municipality_year.merge(
        attrs,
        on=MUNICIPALITY_ID_COLUMN,
        how="left"
    )

# ------------------------------------------------------------------------------
# 6. Reorder columns
# ------------------------------------------------------------------------------

ordered_cols = (
    [MUNICIPALITY_ID_COLUMN]
    + attr_cols
    + [
        "year",
        "months_observed",
        "rows_observed",
        "zero_months",
        "real_zero_months",
        "share_zero_months",
        "share_real_zero_months",
        "valid_pixel_months_total",
        "mean_valid_pixels_per_month",
        "zero_pixels_total",
        "mean_zero_pixels_per_month",
        "max_zero_pixels_month",
        "positive_pixels_total",
        "mean_positive_pixels_per_month",
        "max_positive_pixels_month",
        "invalid_pixels_total",
        "mean_invalid_pixels_per_month",
        "max_invalid_pixels_month",
        "share_zero_pixels",
        "share_positive_pixels",
        "count_zero_positive_gap_sum",
        "count_zero_positive_gap_max_abs",
        "mean_value",
        "min_value",
        "max_value"
    ]
)

ordered_cols = [
    c for c in ordered_cols
    if c in pixel_counts_by_municipality_year.columns
]

remaining_cols = [
    c for c in pixel_counts_by_municipality_year.columns
    if c not in ordered_cols
]

pixel_counts_by_municipality_year = pixel_counts_by_municipality_year[
    ordered_cols + remaining_cols
]

# ------------------------------------------------------------------------------
# 7. Save and display
# ------------------------------------------------------------------------------

pixel_counts_by_municipality_year.to_csv(ZERO_OUTPUT_CSV, index=False)

print(f"Saved pixel-count diagnostics table to: {ZERO_OUTPUT_CSV}")
print("Shape:", pixel_counts_by_municipality_year.shape)

if "EXPECTED_UNITS" in globals():
    print(f"Expected municipalities: {EXPECTED_UNITS:,}")
    print(
        "Municipalities in diagnostics:",
        pixel_counts_by_municipality_year[MUNICIPALITY_ID_COLUMN].nunique()
    )

display(pixel_counts_by_municipality_year.head(20))

print("\nSummary of zero_months:")
display(pixel_counts_by_municipality_year["zero_months"].describe())

if "share_zero_pixels" in pixel_counts_by_municipality_year.columns:
    print("\nSummary of share_zero_pixels:")
    display(pixel_counts_by_municipality_year["share_zero_pixels"].describe())

if "share_positive_pixels" in pixel_counts_by_municipality_year.columns:
    print("\nSummary of share_positive_pixels:")
    display(pixel_counts_by_municipality_year["share_positive_pixels"].describe())

if "count_zero_positive_gap_max_abs" in pixel_counts_by_municipality_year.columns:
    print("\nConsistency check: max absolute count gap by municipality-year")
    display(pixel_counts_by_municipality_year["count_zero_positive_gap_max_abs"].describe())

In [ ]:
# ==============================================================================
# COUNT NaN / NO-VALID-PIXEL / INVALID-PIXEL VALUES BY MUNICIPALITY-YEAR
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------------------------

VALUE_COLUMN = "sum"

# Municipality identifier for Bolivia.
# This should normally be ID_ADMIN, created from asdf_id.
MUNICIPALITY_ID_COLUMN = ID_COL if "ID_COL" in globals() else "ID_ADMIN"

NAN_OUTPUT_CSV = OUTPUT_DIR / "nan_invalid_pixel_diagnostics_by_municipality_year.csv"
NAN_OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------------------------

if "df_final" in globals():
    df_nan = df_final.copy()
    print("Using df_final from memory.")
elif "df_year_stats" in globals():
    df_nan = df_year_stats.copy()
    print("Using df_year_stats from memory.")
else:
    checkpoint_files = sorted(CHECKPOINT_DIR.glob("*.parquet"))

    if not checkpoint_files:
        raise FileNotFoundError(
            "No df_final, no df_year_stats, and no yearly Parquet checkpoints found."
        )

    print(f"Loading {len(checkpoint_files)} yearly checkpoints...")
    df_nan = pd.concat(
        [pd.read_parquet(path) for path in checkpoint_files],
        ignore_index=True
    )

print("Data shape:", df_nan.shape)

# ------------------------------------------------------------------------------
# 3. Validate required columns
# ------------------------------------------------------------------------------

required_columns = [
    MUNICIPALITY_ID_COLUMN,
    "year",
    "month",
    VALUE_COLUMN,
    "count"
]

missing_columns = [col for col in required_columns if col not in df_nan.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

optional_columns = [
    "count_zeros",
    "count_positive",
    "count_invalid"
]

available_optional_columns = [
    col for col in optional_columns
    if col in df_nan.columns
]

df_nan["year"] = pd.to_numeric(df_nan["year"], errors="coerce").astype("Int64")
df_nan["month"] = pd.to_numeric(df_nan["month"], errors="coerce").astype("Int64")
df_nan[VALUE_COLUMN] = pd.to_numeric(df_nan[VALUE_COLUMN], errors="coerce")
df_nan["count"] = pd.to_numeric(df_nan["count"], errors="coerce")

for col in available_optional_columns:
    df_nan[col] = pd.to_numeric(df_nan[col], errors="coerce")

# ------------------------------------------------------------------------------
# 4. Build NaN / invalid indicators
# ------------------------------------------------------------------------------

df_nan["value_is_nan"] = df_nan[VALUE_COLUMN].isna()

# Month with no valid raster pixels inside the municipality.
df_nan["no_valid_pixel_month"] = df_nan["count"].fillna(0).eq(0)

# Month where custom invalid-pixel count is positive.
# Note: count_invalid is diagnostic because masked values may include pixels
# outside the polygon window used internally by rasterstats.
if "count_invalid" in df_nan.columns:
    df_nan["has_invalid_pixels"] = df_nan["count_invalid"].fillna(0).gt(0)
else:
    df_nan["has_invalid_pixels"] = False

# Consistency check for valid pixel classes.
if all(c in df_nan.columns for c in ["count", "count_zeros", "count_positive"]):
    df_nan["count_zero_positive_gap"] = (
        df_nan["count"]
        - df_nan["count_zeros"]
        - df_nan["count_positive"]
    )
else:
    df_nan["count_zero_positive_gap"] = np.nan

# ------------------------------------------------------------------------------
# 5. Count NaN / invalid values by municipality-year
# ------------------------------------------------------------------------------

id_cols = [MUNICIPALITY_ID_COLUMN, "year"]

# Keep useful Bolivia administrative columns if available.
possible_attr_cols = [
    "asdf_id",
    "mun_id",
    "mun_name",
    "municipality_id",
    "municipality_name",
    "municipio_id",
    "municipio",
    "dep_id",
    "dep_name",
    "department_id",
    "department_name",
    "departamento",
    "name",
    "NAME"
]

# Also keep any previously selected administrative attributes.
if "ATTRIBUTE_COLS" in globals():
    possible_attr_cols = possible_attr_cols + [
        c for c in ATTRIBUTE_COLS
        if c not in possible_attr_cols and c != MUNICIPALITY_ID_COLUMN
    ]

attr_cols = [
    c for c in possible_attr_cols
    if c in df_nan.columns and c != MUNICIPALITY_ID_COLUMN
]

agg_dict = {
    "months_observed": ("month", "nunique"),
    "rows_observed": ("month", "size"),
    "nan_months": ("value_is_nan", "sum"),
    "no_valid_pixel_months": ("no_valid_pixel_month", "sum"),
    "valid_pixel_months_total": ("count", "sum"),
    "mean_valid_pixels_per_month": ("count", "mean"),
    "min_valid_pixels_month": ("count", "min"),
    "max_valid_pixels_month": ("count", "max"),
    "mean_value": (VALUE_COLUMN, "mean"),
    "min_value": (VALUE_COLUMN, "min"),
    "max_value": (VALUE_COLUMN, "max"),
    "count_zero_positive_gap_sum": ("count_zero_positive_gap", "sum"),
    "count_zero_positive_gap_max_abs": (
        "count_zero_positive_gap",
        lambda x: np.nanmax(np.abs(x)) if x.notna().any() else np.nan
    )
}

if "count_invalid" in df_nan.columns:
    agg_dict.update({
        "invalid_pixel_months": ("has_invalid_pixels", "sum"),
        "invalid_pixels_total": ("count_invalid", "sum"),
        "mean_invalid_pixels_per_month": ("count_invalid", "mean"),
        "max_invalid_pixels_month": ("count_invalid", "max")
    })

if "count_zeros" in df_nan.columns:
    agg_dict.update({
        "zero_pixels_total": ("count_zeros", "sum"),
        "mean_zero_pixels_per_month": ("count_zeros", "mean")
    })

if "count_positive" in df_nan.columns:
    agg_dict.update({
        "positive_pixels_total": ("count_positive", "sum"),
        "mean_positive_pixels_per_month": ("count_positive", "mean")
    })

nan_by_municipality_year = (
    df_nan
    .groupby(id_cols, as_index=False)
    .agg(**agg_dict)
)

nan_by_municipality_year["share_nan_months"] = (
    nan_by_municipality_year["nan_months"] /
    nan_by_municipality_year["rows_observed"]
)

nan_by_municipality_year["share_no_valid_pixel_months"] = (
    nan_by_municipality_year["no_valid_pixel_months"] /
    nan_by_municipality_year["rows_observed"]
)

if "invalid_pixel_months" in nan_by_municipality_year.columns:
    nan_by_municipality_year["share_invalid_pixel_months"] = (
        nan_by_municipality_year["invalid_pixel_months"] /
        nan_by_municipality_year["rows_observed"]
    )

if "invalid_pixels_total" in nan_by_municipality_year.columns:
    nan_by_municipality_year["invalid_per_valid_pixel_ratio"] = np.where(
        nan_by_municipality_year["valid_pixel_months_total"].gt(0),
        nan_by_municipality_year["invalid_pixels_total"] /
        nan_by_municipality_year["valid_pixel_months_total"],
        np.nan
    )

# Add attributes back.
if attr_cols:
    attrs = (
        df_nan[[MUNICIPALITY_ID_COLUMN] + attr_cols]
        .drop_duplicates(subset=[MUNICIPALITY_ID_COLUMN])
    )

    nan_by_municipality_year = nan_by_municipality_year.merge(
        attrs,
        on=MUNICIPALITY_ID_COLUMN,
        how="left"
    )

# ------------------------------------------------------------------------------
# 6. Reorder columns
# ------------------------------------------------------------------------------

ordered_cols = (
    [MUNICIPALITY_ID_COLUMN]
    + attr_cols
    + [
        "year",
        "months_observed",
        "rows_observed",
        "nan_months",
        "share_nan_months",
        "no_valid_pixel_months",
        "share_no_valid_pixel_months",
        "valid_pixel_months_total",
        "mean_valid_pixels_per_month",
        "min_valid_pixels_month",
        "max_valid_pixels_month",
        "invalid_pixel_months",
        "share_invalid_pixel_months",
        "invalid_pixels_total",
        "mean_invalid_pixels_per_month",
        "max_invalid_pixels_month",
        "invalid_per_valid_pixel_ratio",
        "zero_pixels_total",
        "mean_zero_pixels_per_month",
        "positive_pixels_total",
        "mean_positive_pixels_per_month",
        "count_zero_positive_gap_sum",
        "count_zero_positive_gap_max_abs",
        "mean_value",
        "min_value",
        "max_value"
    ]
)

ordered_cols = [
    c for c in ordered_cols
    if c in nan_by_municipality_year.columns
]

remaining_cols = [
    c for c in nan_by_municipality_year.columns
    if c not in ordered_cols
]

nan_by_municipality_year = nan_by_municipality_year[
    ordered_cols + remaining_cols
]

# ------------------------------------------------------------------------------
# 7. Save and display
# ------------------------------------------------------------------------------

nan_by_municipality_year.to_csv(NAN_OUTPUT_CSV, index=False)

print(f"Saved NaN / invalid-pixel diagnostics table to: {NAN_OUTPUT_CSV}")
print("Shape:", nan_by_municipality_year.shape)

if "EXPECTED_UNITS" in globals():
    print(f"Expected municipalities: {EXPECTED_UNITS:,}")
    print(
        "Municipalities in diagnostics:",
        nan_by_municipality_year[MUNICIPALITY_ID_COLUMN].nunique()
    )

display(nan_by_municipality_year.head(20))

print("\nSummary of nan_months:")
display(nan_by_municipality_year["nan_months"].describe())

print("\nSummary of no_valid_pixel_months:")
display(nan_by_municipality_year["no_valid_pixel_months"].describe())

if "invalid_pixel_months" in nan_by_municipality_year.columns:
    print("\nSummary of invalid_pixel_months:")
    display(nan_by_municipality_year["invalid_pixel_months"].describe())

if "invalid_per_valid_pixel_ratio" in nan_by_municipality_year.columns:
    print("\nSummary of invalid_per_valid_pixel_ratio:")
    display(nan_by_municipality_year["invalid_per_valid_pixel_ratio"].describe())

print("\nMunicipality-years with at least one NaN month:")
display(
    nan_by_municipality_year[
        nan_by_municipality_year["nan_months"] > 0
    ].sort_values(
        ["nan_months", "year"],
        ascending=[False, True]
    ).head(30)
)

print("\nMunicipality-years with at least one no-valid-pixel month:")
display(
    nan_by_municipality_year[
        nan_by_municipality_year["no_valid_pixel_months"] > 0
    ].sort_values(
        ["no_valid_pixel_months", "year"],
        ascending=[False, True]
    ).head(30)
)

## Step 7. Remaining years lopp

In [ ]:
# ==============================================================================
# CELL 11 — PRODUCTION LOOP FOR REMAINING YEARS: 1993–2024
# ==============================================================================
# This cell processes the remaining years after the successful 1992 test.
#
# Strategy:
# - Years are processed sequentially.
# - Months inside each year are processed in parallel.
# - Each year is saved as a yearly checkpoint.
# - Temporary extracted rasters are deleted after each year.
# - Existing checkpoints are skipped automatically.
#
# Expected full-year rows for Bolivia:
# number of Bolivian municipalities × 12 months.
#
# With the ds4bolivia optimized municipal map:
# 339 municipalities × 12 months = 4,068 rows per full year.

# ------------------------------------------------------------------------------
# 1. Production-loop configuration
# ------------------------------------------------------------------------------

REMAINING_START_YEAR = 1993
REMAINING_END_YEAR = END_YEAR

# For a short mini-production test, temporarily use:
# REMAINING_END_YEAR = 1995

SKIP_EXISTING_CHECKPOINTS = True
SAVE_YEAR_CHECKPOINT = True

CLEAN_EXTRACTED_RASTERS_AFTER_PROCESS = True
DELETE_YEAR_ZIP_AFTER_PROCESS = True

# Keep this True during long production runs.
CONTINUE_ON_YEAR_FAILURE = True

print("Production loop configuration")
print("-" * 70)
print(f"Country                       : {COUNTRY_NAME}")
print(f"Years to process              : {REMAINING_START_YEAR}-{REMAINING_END_YEAR}")
print(f"Skip existing checkpoints     : {SKIP_EXISTING_CHECKPOINTS}")
print(f"Processing mode               : {PROCESSING_MODE}")
print(f"Delete annual ZIP after year  : {DELETE_YEAR_ZIP_AFTER_PROCESS}")
print(f"Clean extracted rasters       : {CLEAN_EXTRACTED_RASTERS_AFTER_PROCESS}")
print(f"Municipalities                : {len(gdf_admin):,}")

if "EXPECTED_UNITS" in globals():
    print(f"Expected municipalities       : {EXPECTED_UNITS:,}")
    print(f"Expected rows per full year   : {EXPECTED_UNITS * 12:,}")

print("-" * 70)

# ------------------------------------------------------------------------------
# 2. Required objects check
# ------------------------------------------------------------------------------

required_objects = [
    "gdf_admin",
    "ATTRIBUTE_COLS",
    "geometry_col",
    "annual_zip_files",
    "YEAR_ZIP_TEMPLATE",
    "RASTER_REPO_ID",
    "RASTER_REPO_TYPE",
    "RASTER_TOKEN",
    "MODE_SETTINGS",
    "PROCESSING_MODE",
    "ZONAL_STATS",
    "CUSTOM_ZONAL_STATS",
    "ALL_ZONAL_OUTPUT_STATS",
    "ALL_TOUCHED",
    "CHECKPOINT_DIR",
    "TEMP_DIR",
    "OUTPUT_DIR",
    "COUNTRY_NAME",
    "PRODUCT_NAME",
    "PROJECT_NAME",
    "ID_COL",
    "FIRST_AVAILABLE_MONTH_BY_YEAR",
    "total_cores",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise RuntimeError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_objects)
    )

if COUNTRY_NAME.lower() != "bolivia":
    print(
        f"Warning: COUNTRY_NAME is currently '{COUNTRY_NAME}'. "
        "This production loop is configured for the Bolivia municipal case."
    )

if ID_COL not in gdf_admin.columns:
    raise ValueError(
        f"ID column '{ID_COL}' was not found in gdf_admin. "
        "Run the Bolivia feature-standardization cell before this production loop."
    )

if "EXPECTED_UNITS" in globals():
    if len(gdf_admin) != EXPECTED_UNITS:
        print(
            f"Warning: expected {EXPECTED_UNITS:,} municipalities, "
            f"but gdf_admin has {len(gdf_admin):,} features."
        )

    if gdf_admin[ID_COL].nunique() != EXPECTED_UNITS:
        print(
            f"Warning: expected {EXPECTED_UNITS:,} unique IDs, "
            f"but found {gdf_admin[ID_COL].nunique():,}."
        )

# Safety check: avoid accidental old country paths/names.
path_safety_text = " ".join([
    str(PROJECT_NAME).lower(),
    str(CHECKPOINT_DIR).lower(),
    str(OUTPUT_DIR).lower(),
    str(TEMP_DIR).lower()
])

if "peru" in path_safety_text or "indonesia" in path_safety_text:
    raise ValueError(
        "This production loop still seems to point to Peru or Indonesia paths/names. "
        "Check PROJECT_NAME, CHECKPOINT_DIR, OUTPUT_DIR, TEMP_DIR, and BASE_DIR before continuing."
    )

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def parse_year_month_from_name(name):
    """
    Extract year and month from a filename/path.

    Supported examples:
    - 199204.tif
    - 1992_04.tif
    - 1992-04.tif
    - SVDNB_npp_simulated_199204.tif
    - folder/1992/SVDNB_npp_simulated_199204.tif
    """
    text = str(name).replace("/", "_")
    base = Path(name).name

    patterns = [
        r"(?P<year>19\d{2}|20\d{2})[_-]?(?P<month>0[1-9]|1[0-2])",
        r"(?P<year>19\d{2}|20\d{2}).*?month[_-]?(?P<month>0[1-9]|1[0-2])",
        r"(?P<year>19\d{2}|20\d{2}).*?m[_-]?(?P<month>0[1-9]|1[0-2])",
    ]

    for target in [base, text]:
        for pattern in patterns:
            match = re.search(pattern, target, flags=re.IGNORECASE)
            if match:
                return int(match.group("year")), int(match.group("month"))

    return None, None


def choose_month_workers(number_of_monthly_rasters):
    """
    Choose a safe number of parallel workers for month-level processing.
    """
    settings = MODE_SETTINGS[PROCESSING_MODE]
    current_available_ram_gb = psutil.virtual_memory().available / (1024 ** 3)

    workers = min(
        number_of_monthly_rasters,
        max(1, total_cores - 2),
        max(1, int(current_available_ram_gb / settings["ram_per_worker_gb"])),
        settings["max_month_workers"]
    )

    return max(1, workers)


def get_year_checkpoint_paths(year):
    """
    Return the expected checkpoint paths for a given year.
    """
    parquet_path = CHECKPOINT_DIR / f"{PROJECT_NAME}_{year}.parquet"
    csv_path = CHECKPOINT_DIR / f"{PROJECT_NAME}_{year}.csv"
    return parquet_path, csv_path


def process_single_month_for_year(row, gdf_for_stats_year, year_zip_filename):
    """
    Calculate zonal statistics for one monthly raster.
    """
    member = row["member"]
    year = int(row["year"])
    month = int(row["month"])
    raster_path = row["raster_path"]

    try:
        with rasterio.open(raster_path) as src:
            month_nodata = src.nodata

        stats = zonal_stats(
            vectors=gdf_for_stats_year.geometry,
            raster=raster_path,
            stats=ZONAL_STATS,
            add_stats=CUSTOM_ZONAL_STATS,
            nodata=month_nodata,
            all_touched=ALL_TOUCHED,
            geojson_out=False
        )

        df_stats = pd.DataFrame(stats)
        df_attrs = gdf_for_stats_year[ATTRIBUTE_COLS].reset_index(drop=True).copy()

        df_month = pd.concat([df_attrs, df_stats], axis=1)

        df_month["country"] = COUNTRY_NAME
        df_month["year"] = year
        df_month["month"] = month
        df_month["product"] = PRODUCT_NAME
        df_month["source_member"] = member
        df_month["source_zip"] = year_zip_filename

        print(f"Finished {year}-{month:02d}: {Path(member).name}")

        return df_month

    except Exception as e:
        print(f"Failed {year}-{month:02d}: {member}")
        print(e)
        traceback.print_exc()
        return None


def process_single_year_production(target_year):
    """
    Download, extract, process, checkpoint, and clean one annual ZIP.
    """
    print("\n" + "=" * 80)
    print(f"PROCESSING YEAR: {target_year}")
    print("=" * 80)

    year_checkpoint_parquet, year_checkpoint_csv = get_year_checkpoint_paths(target_year)

    if SKIP_EXISTING_CHECKPOINTS and year_checkpoint_parquet.exists():
        print(f"Skipping {target_year}: checkpoint already exists.")
        print(f"Checkpoint: {year_checkpoint_parquet}")
        return {
            "year": target_year,
            "status": "skipped_existing_checkpoint",
            "rows": None,
            "months": None,
            "checkpoint": str(year_checkpoint_parquet)
        }

    year_zip_filename = YEAR_ZIP_TEMPLATE.format(year=target_year)

    if year_zip_filename not in annual_zip_files:
        raise FileNotFoundError(
            f"ZIP for year {target_year} not found in annual_zip_files: {year_zip_filename}"
        )

    # --------------------------------------------------------------------------
    # Download annual ZIP
    # --------------------------------------------------------------------------

    year_temp_dir = TEMP_DIR / f"production_{target_year}"
    year_download_dir = year_temp_dir / "download"
    year_extract_dir = year_temp_dir / "extract"

    if year_temp_dir.exists():
        shutil.rmtree(year_temp_dir, ignore_errors=True)

    year_download_dir.mkdir(parents=True, exist_ok=True)
    year_extract_dir.mkdir(parents=True, exist_ok=True)

    year_zip_path = hf_hub_download(
        repo_id=RASTER_REPO_ID,
        filename=year_zip_filename,
        repo_type=RASTER_REPO_TYPE,
        token=RASTER_TOKEN,
        local_dir=str(year_download_dir)
    )

    year_zip_path = Path(year_zip_path)

    print("-" * 70)
    print(f"Downloaded ZIP : {year_zip_path}")
    print(f"ZIP size       : {year_zip_path.stat().st_size / (1024 ** 3):.2f} GB")
    print("-" * 70)

    # --------------------------------------------------------------------------
    # Identify monthly rasters inside ZIP
    # --------------------------------------------------------------------------

    with zipfile.ZipFile(year_zip_path, "r") as zf:
        zip_members = zf.namelist()

    monthly_raster_members = [
        m for m in zip_members
        if m.lower().endswith((".tif", ".tiff")) and not m.endswith("/")
    ]

    monthly_records = []

    for member in monthly_raster_members:
        year, month = parse_year_month_from_name(member)

        if year == target_year:
            monthly_records.append({
                "member": member,
                "year": year,
                "month": month
            })

    year_months_df = pd.DataFrame(
        monthly_records,
        columns=["member", "year", "month"]
    )

    if year_months_df.empty:
        raise RuntimeError(
            f"No monthly rasters were parsed for year {target_year}."
        )

    year_months_df = (
        year_months_df
        .sort_values(["year", "month", "member"])
        .reset_index(drop=True)
    )

    expected_months = list(
        range(FIRST_AVAILABLE_MONTH_BY_YEAR.get(target_year, 1), 13)
    )

    found_months = sorted(year_months_df["month"].unique().tolist())
    missing_months = sorted(set(expected_months) - set(found_months))

    print("Monthly rasters selected:")
    display(year_months_df)

    print("-" * 70)
    print(f"Expected months : {expected_months}")
    print(f"Found months    : {found_months}")
    print(f"Missing months  : {missing_months}")
    print("-" * 70)

    if missing_months:
        raise RuntimeError(
            f"Missing monthly rasters for {target_year}: {missing_months}"
        )

    # --------------------------------------------------------------------------
    # Extract monthly rasters
    # --------------------------------------------------------------------------

    print(f"Extracting {len(year_months_df)} monthly rasters...")

    with zipfile.ZipFile(year_zip_path, "r") as zf:
        for member in year_months_df["member"].tolist():
            zf.extract(member, path=year_extract_dir)

    year_months_df["raster_path"] = year_months_df["member"].apply(
        lambda m: str(year_extract_dir / m)
    )

    missing_extracted = [
        p for p in year_months_df["raster_path"].tolist()
        if not Path(p).exists()
    ]

    if missing_extracted:
        raise FileNotFoundError(
            "Some monthly rasters were not extracted correctly:\n"
            + "\n".join(missing_extracted[:10])
        )

    print("Extraction completed.")

    # --------------------------------------------------------------------------
    # CRS check and polygon preparation
    # --------------------------------------------------------------------------

    sample_raster_path = year_months_df.loc[0, "raster_path"]

    with rasterio.open(sample_raster_path) as src:
        raster_crs = src.crs
        raster_nodata = src.nodata
        raster_bounds = src.bounds
        raster_res = src.res
        raster_shape = (src.height, src.width)

    print("-" * 70)
    print("Sample raster metadata")
    print(f"Sample raster : {sample_raster_path}")
    print(f"Raster CRS    : {raster_crs}")
    print(f"Raster nodata : {raster_nodata}")
    print(f"Raster bounds : {raster_bounds}")
    print(f"Raster res    : {raster_res}")
    print(f"Raster shape  : {raster_shape[0]:,} rows x {raster_shape[1]:,} cols")
    print("-" * 70)

    if raster_crs is None:
        raise ValueError(
            f"Raster CRS is None for {target_year}. Stop and define CRS manually."
        )

    if gdf_admin.crs != raster_crs:
        print(f"Reprojecting Bolivia municipal polygons from {gdf_admin.crs} to {raster_crs}...")
        gdf_for_stats_year = (
            gdf_admin[ATTRIBUTE_COLS + [geometry_col]]
            .to_crs(raster_crs)
            .copy()
        )
    else:
        print("Vector CRS already matches raster CRS. No reprojection needed.")
        gdf_for_stats_year = gdf_admin[ATTRIBUTE_COLS + [geometry_col]].copy()

    gdf_for_stats_year = gdf_for_stats_year.set_geometry(geometry_col)

    print(f"Municipalities ready for zonal statistics: {len(gdf_for_stats_year):,}")

    # --------------------------------------------------------------------------
    # Month-level parallel processing
    # --------------------------------------------------------------------------

    month_workers = choose_month_workers(len(year_months_df))

    print("-" * 70)
    print(f"Processing mode          : {PROCESSING_MODE}")
    print(f"Monthly rasters          : {len(year_months_df)}")
    print(f"CPU logical cores        : {total_cores}")
    print(f"Available RAM now        : {psutil.virtual_memory().available / (1024 ** 3):.2f} GB")
    print(f"Monthly workers selected : {month_workers}")
    print("-" * 70)

    monthly_rows = year_months_df.to_dict("records")

    monthly_results = Parallel(
        n_jobs=month_workers,
        backend="loky",
        verbose=5
    )(
        delayed(process_single_month_for_year)(
            row,
            gdf_for_stats_year,
            year_zip_filename
        )
        for row in monthly_rows
    )

    valid_monthly_results = [
        df for df in monthly_results
        if df is not None
    ]

    if not valid_monthly_results:
        raise RuntimeError(f"No valid monthly results were generated for {target_year}.")

    df_year = pd.concat(valid_monthly_results, ignore_index=True)

    # --------------------------------------------------------------------------
    # Clean, order, validate
    # --------------------------------------------------------------------------

    stat_cols = [
        c for c in ALL_ZONAL_OUTPUT_STATS
        if c in df_year.columns
    ]

    preferred_cols = (
        [ID_COL]
        + [c for c in ATTRIBUTE_COLS if c != ID_COL and c in df_year.columns]
        + ["country", "year", "month", "product"]
        + stat_cols
        + ["source_member", "source_zip"]
    )

    remaining_cols = [
        c for c in df_year.columns
        if c not in preferred_cols
    ]

    df_year = df_year[preferred_cols + remaining_cols]

    df_year = df_year.sort_values(
        ["year", "month", ID_COL]
    ).reset_index(drop=True)

    expected_rows = len(gdf_admin) * len(year_months_df)

    print("-" * 70)
    print(f"Completed year   : {target_year}")
    print(f"Months processed : {len(year_months_df)}")
    print(f"Municipalities   : {len(gdf_admin):,}")
    print(f"Expected rows    : {expected_rows:,}")
    print(f"Actual rows      : {len(df_year):,}")
    print("-" * 70)

    if len(df_year) != expected_rows:
        print("Warning: row count does not match expected rows.")
    else:
        print("Good: row count matches municipalities × months.")

    if "EXPECTED_UNITS" in globals():
        expected_rows_from_config = EXPECTED_UNITS * len(year_months_df)

        print("-" * 70)
        print("Configured Bolivia expectation")
        print(f"Expected units from config : {EXPECTED_UNITS:,}")
        print(f"Expected rows from config  : {expected_rows_from_config:,}")
        print("-" * 70)

        if len(df_year) != expected_rows_from_config:
            print("Warning: row count does not match EXPECTED_UNITS × months.")
        else:
            print("Good: row count matches EXPECTED_UNITS × months.")

    month_row_check = (
        df_year
        .groupby(["year", "month"])
        .size()
        .reset_index(name="rows")
    )

    print("Rows by year-month:")
    display(month_row_check)

    missing_stats = (
        df_year[stat_cols]
        .isna()
        .sum()
        .reset_index(name="missing_values")
        .rename(columns={"index": "variable"})
    )

    print("Missing values in main statistics:")
    display(missing_stats)

    # --------------------------------------------------------------------------
    # Pixel-count consistency diagnostics
    # --------------------------------------------------------------------------

    if all(c in df_year.columns for c in ["count", "count_zeros", "count_positive"]):
        df_year["count_zero_positive_gap"] = (
            df_year["count"]
            - df_year["count_zeros"]
            - df_year["count_positive"]
        )

        gap_summary = (
            df_year["count_zero_positive_gap"]
            .describe()
            .to_frame(name="count_zero_positive_gap")
        )

        print("Check: count - count_zeros - count_positive")
        display(gap_summary)

        nonzero_gap_rows = int((df_year["count_zero_positive_gap"] != 0).sum())

        if nonzero_gap_rows == 0:
            print("Good: count equals count_zeros + count_positive for all rows.")
        else:
            print(
                f"Warning: {nonzero_gap_rows:,} rows have "
                "count != count_zeros + count_positive."
            )

    # --------------------------------------------------------------------------
    # Save checkpoint
    # --------------------------------------------------------------------------

    if SAVE_YEAR_CHECKPOINT:
        df_year.to_parquet(year_checkpoint_parquet, index=False)
        df_year.to_csv(year_checkpoint_csv, index=False)

        print("-" * 70)
        print("Yearly checkpoint saved:")
        print(f"Parquet : {year_checkpoint_parquet}")
        print(f"CSV     : {year_checkpoint_csv}")
        print("-" * 70)

    # --------------------------------------------------------------------------
    # Clean temporary files
    # --------------------------------------------------------------------------

    if CLEAN_EXTRACTED_RASTERS_AFTER_PROCESS and year_extract_dir.exists():
        shutil.rmtree(year_extract_dir, ignore_errors=True)
        print(f"Deleted extracted raster directory: {year_extract_dir}")

    if DELETE_YEAR_ZIP_AFTER_PROCESS and year_zip_path.exists():
        year_zip_path.unlink()
        print(f"Deleted annual ZIP: {year_zip_path}")

    try:
        shutil.rmtree(year_temp_dir, ignore_errors=True)
        print(f"Deleted year temp directory: {year_temp_dir}")
    except Exception:
        pass

    gc.collect()

    return {
        "year": target_year,
        "status": "processed",
        "rows": len(df_year),
        "months": len(year_months_df),
        "checkpoint": str(year_checkpoint_parquet)
    }

# ------------------------------------------------------------------------------
# 4. Run production loop for remaining years
# ------------------------------------------------------------------------------

years_to_process = list(range(REMAINING_START_YEAR, REMAINING_END_YEAR + 1))

print("Years queued for processing:")
print(years_to_process)

production_log = []

for year in years_to_process:
    try:
        result = process_single_year_production(year)
        production_log.append(result)

    except Exception as e:
        print("\n" + "!" * 80)
        print(f"YEAR FAILED: {year}")
        print(e)
        traceback.print_exc()
        print("!" * 80 + "\n")

        production_log.append({
            "year": year,
            "status": "failed",
            "rows": None,
            "months": None,
            "checkpoint": None,
            "error": str(e)
        })

        if not CONTINUE_ON_YEAR_FAILURE:
            raise

production_log_df = pd.DataFrame(production_log)

country_slug_for_file = globals().get("COUNTRY_SLUG", COUNTRY_NAME.lower())

PRODUCTION_LOG_CSV = OUTPUT_DIR / (
    f"production_loop_log_{country_slug_for_file}_"
    f"{REMAINING_START_YEAR}_{REMAINING_END_YEAR}.csv"
)

production_log_df.to_csv(PRODUCTION_LOG_CSV, index=False)

print("\n" + "=" * 80)
print("PRODUCTION LOOP FINISHED")
print("=" * 80)
display(production_log_df)

print(f"Production log saved to: {PRODUCTION_LOG_CSV}")

## Step 8. Unified dataset

In [ ]:
# ==============================================================================
# CELL 12 — COMPILE FINAL MONTHLY PANEL: BOLIVIA, 1992–2024
# ==============================================================================
# This cell combines all yearly checkpoints into one final monthly panel.
#
# Expected total months:
# - 1992: April-December = 9 months
# - 1993-2024: 32 years × 12 months = 384 months
# - Total = 393 months
#
# Expected total rows for Bolivia:
# - number of Bolivian municipalities × 393 months
# - with the ds4bolivia optimized municipal map:
#   339 municipalities × 393 months = 133,227 rows

print(f"Compiling final monthly panel for {COUNTRY_NAME} from yearly checkpoints...")

# ------------------------------------------------------------------------------
# 1. Required objects check
# ------------------------------------------------------------------------------

required_objects = [
    "CHECKPOINT_DIR",
    "PROJECT_NAME",
    "OUTPUT_PARQUET",
    "OUTPUT_CSV",
    "OUTPUT_ZIP",
    "START_YEAR",
    "END_YEAR",
    "FIRST_AVAILABLE_MONTH_BY_YEAR",
    "COUNTRY_NAME",
    "ID_COL",
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise RuntimeError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_objects)
    )

if COUNTRY_NAME.lower() != "bolivia":
    print(
        f"Warning: COUNTRY_NAME is currently '{COUNTRY_NAME}'. "
        "This compilation cell is configured for the Bolivia municipal case."
    )

if "EXPECTED_UNITS" not in globals():
    if "gdf_admin" in globals():
        EXPECTED_UNITS = len(gdf_admin)
        print(f"EXPECTED_UNITS was not defined. Using len(gdf_admin): {EXPECTED_UNITS:,}")
    else:
        raise RuntimeError(
            "EXPECTED_UNITS is not defined and gdf_admin is not available."
        )

# Safety check: avoid accidental old country paths/names.
path_safety_text = " ".join([
    str(PROJECT_NAME).lower(),
    str(CHECKPOINT_DIR).lower(),
    str(OUTPUT_PARQUET).lower(),
    str(OUTPUT_CSV).lower(),
    str(OUTPUT_ZIP).lower()
])

if "peru" in path_safety_text or "indonesia" in path_safety_text:
    raise ValueError(
        "This final-panel cell still seems to point to Peru or Indonesia paths/names. "
        "Check PROJECT_NAME, CHECKPOINT_DIR, OUTPUT_PARQUET, OUTPUT_CSV, and OUTPUT_ZIP."
    )

# ------------------------------------------------------------------------------
# 2. Locate yearly checkpoint files
# ------------------------------------------------------------------------------

checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(f"{PROJECT_NAME}_*.parquet")
)

if not checkpoint_files:
    raise FileNotFoundError(
        f"No yearly Parquet checkpoints found in: {CHECKPOINT_DIR}"
    )

print(f"Yearly checkpoint files found: {len(checkpoint_files)}")

for path in checkpoint_files[:5]:
    print(" -", path.name)

if len(checkpoint_files) > 10:
    print(" ...")

for path in checkpoint_files[-5:]:
    print(" -", path.name)

# ------------------------------------------------------------------------------
# 3. Load all checkpoints
# ------------------------------------------------------------------------------

df_final = pd.concat(
    [pd.read_parquet(path) for path in checkpoint_files],
    ignore_index=True
)

print("--------------------------------------------------")
print("Final monthly panel loaded.")
print(f"Rows    : {len(df_final):,}")
print(f"Columns : {len(df_final.columns):,}")
print("--------------------------------------------------")

# ------------------------------------------------------------------------------
# 4. Basic type cleaning
# ------------------------------------------------------------------------------

df_final["year"] = pd.to_numeric(df_final["year"], errors="coerce").astype("Int64")
df_final["month"] = pd.to_numeric(df_final["month"], errors="coerce").astype("Int64")

numeric_cols = [
    "count",
    "min",
    "max",
    "median",
    "mean",
    "sum",
    "std",
    "count_zeros",
    "count_positive",
    "count_invalid",
    "count_zero_positive_gap"
]

for col in numeric_cols:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors="coerce")

# Preserve Bolivia municipality identifiers as strings.
# Do not apply Peru-style zfill(6); Bolivia uses ds4bolivia asdf_id.
if ID_COL in df_final.columns:
    df_final[ID_COL] = df_final[ID_COL].astype(str).str.strip()

if "ID_ADMIN" in df_final.columns:
    df_final["ID_ADMIN"] = df_final["ID_ADMIN"].astype(str).str.strip()

if "asdf_id" in df_final.columns:
    df_final["asdf_id"] = df_final["asdf_id"].astype(str).str.strip().str.replace(
        r"\.0$",
        "",
        regex=True
    )

if "mun_id" in df_final.columns:
    df_final["mun_id"] = df_final["mun_id"].astype(str).str.strip().str.replace(
        r"\.0$",
        "",
        regex=True
    )

if "dep_id" in df_final.columns:
    df_final["dep_id"] = df_final["dep_id"].astype(str).str.strip().str.replace(
        r"\.0$",
        "",
        regex=True
    )

# ------------------------------------------------------------------------------
# 5. Sort final panel
# ------------------------------------------------------------------------------

if ID_COL not in df_final.columns:
    raise ValueError(
        f"ID column '{ID_COL}' was not found in df_final. "
        "Check the yearly checkpoint files."
    )

df_final = df_final.sort_values(
    ["year", "month", ID_COL]
).reset_index(drop=True)

# ------------------------------------------------------------------------------
# 6. Build expected year-month structure
# ------------------------------------------------------------------------------

expected_year_month_records = []

for year in range(START_YEAR, END_YEAR + 1):
    first_month = FIRST_AVAILABLE_MONTH_BY_YEAR.get(year, 1)

    for month in range(first_month, 13):
        expected_year_month_records.append({
            "year": year,
            "month": month
        })

expected_year_months_df = pd.DataFrame(expected_year_month_records)

expected_total_months = len(expected_year_months_df)
expected_total_rows = EXPECTED_UNITS * expected_total_months

actual_year_months_df = (
    df_final[["year", "month"]]
    .drop_duplicates()
    .sort_values(["year", "month"])
    .reset_index(drop=True)
)

actual_total_months = len(actual_year_months_df)
actual_total_rows = len(df_final)

actual_units = df_final[ID_COL].nunique(dropna=True)

print("Validation summary")
print("-" * 70)
print(f"Country              : {COUNTRY_NAME}")
print(f"Unit type            : Bolivian municipalities")
print(f"ID column            : {ID_COL}")
print(f"Expected units       : {EXPECTED_UNITS:,}")
print(f"Actual units         : {actual_units:,}")
print(f"Expected months      : {expected_total_months:,}")
print(f"Actual months        : {actual_total_months:,}")
print(f"Expected total rows  : {expected_total_rows:,}")
print(f"Actual total rows    : {actual_total_rows:,}")
print("-" * 70)

if actual_units != EXPECTED_UNITS:
    print("Warning: number of unique municipalities does not match EXPECTED_UNITS.")
else:
    print("Good: unique municipality count matches EXPECTED_UNITS.")

if actual_total_months != expected_total_months:
    print("Warning: number of year-month combinations does not match expected value.")
else:
    print("Good: expected number of year-month combinations found.")

if actual_total_rows != expected_total_rows:
    print("Warning: total row count does not match expected value.")
else:
    print("Good: total row count matches municipalities × months.")

# ------------------------------------------------------------------------------
# 7. Validate rows per year-month
# ------------------------------------------------------------------------------

rows_by_month = (
    df_final
    .groupby(["year", "month"])
    .size()
    .reset_index(name="rows")
)

bad_months = rows_by_month[
    rows_by_month["rows"] != EXPECTED_UNITS
].copy()

print("\nRows by year-month check")
print("-" * 70)

if bad_months.empty:
    print(f"Good: every year-month has exactly {EXPECTED_UNITS:,} municipality rows.")
else:
    print("Warning: some year-months do not have the expected number of rows.")
    display(bad_months)

display(rows_by_month.head(15))
display(rows_by_month.tail(15))

# ------------------------------------------------------------------------------
# 8. Validate missing years/months
# ------------------------------------------------------------------------------

expected_pairs = set(
    map(tuple, expected_year_months_df[["year", "month"]].to_numpy())
)

actual_pairs = set(
    map(tuple, actual_year_months_df[["year", "month"]].to_numpy())
)

missing_pairs = sorted(expected_pairs - actual_pairs)
extra_pairs = sorted(actual_pairs - expected_pairs)

years_found = sorted(df_final["year"].dropna().unique().tolist())

print("\nYears found")
print("-" * 70)
print(years_found)

expected_years = list(range(START_YEAR, END_YEAR + 1))
missing_years = sorted(set(expected_years) - set(years_found))

if missing_years:
    print("Warning: missing years:")
    print(missing_years)
else:
    print(f"Good: all years {START_YEAR}-{END_YEAR} are present.")

if missing_pairs:
    print("\nWarning: missing year-month pairs:")
    display(pd.DataFrame(missing_pairs, columns=["year", "month"]).head(50))
else:
    print("\nGood: no missing year-month pairs.")

if extra_pairs:
    print("\nWarning: extra year-month pairs found:")
    display(pd.DataFrame(extra_pairs, columns=["year", "month"]).head(50))

months_by_year = (
    df_final
    .groupby("year")["month"]
    .nunique()
    .reset_index(name="months_observed")
)

print("\nMonths observed by year")
display(months_by_year)

# ------------------------------------------------------------------------------
# 9. Missing values in main statistics
# ------------------------------------------------------------------------------

main_stats = [
    c for c in [
        "count",
        "min",
        "max",
        "median",
        "mean",
        "sum",
        "std",
        "count_zeros",
        "count_positive",
        "count_invalid",
        "count_zero_positive_gap"
    ]
    if c in df_final.columns
]

missing_summary = (
    df_final[main_stats]
    .isna()
    .sum()
    .reset_index(name="missing_values")
    .rename(columns={"index": "variable"})
)

print("\nMissing values in main statistics")
print("-" * 70)
display(missing_summary)

# ------------------------------------------------------------------------------
# 10. Pixel-count consistency check
# ------------------------------------------------------------------------------

if all(c in df_final.columns for c in ["count", "count_zeros", "count_positive"]):
    df_final["count_zero_positive_gap"] = (
        df_final["count"]
        - df_final["count_zeros"]
        - df_final["count_positive"]
    )

    print("\nPixel-count consistency check")
    print("-" * 70)
    print("Checking: count - count_zeros - count_positive")

    display(
        df_final["count_zero_positive_gap"]
        .describe()
        .to_frame(name="count_zero_positive_gap")
    )

    nonzero_gap_rows = int((df_final["count_zero_positive_gap"] != 0).sum())

    if nonzero_gap_rows == 0:
        print("Good: count equals count_zeros + count_positive for all rows.")
    else:
        print(
            f"Warning: {nonzero_gap_rows:,} rows have "
            "count != count_zeros + count_positive."
        )

# ------------------------------------------------------------------------------
# 11. Basic panel preview
# ------------------------------------------------------------------------------

print("\nFinal panel preview")
print("-" * 70)
display(df_final.head())

print("\nFinal panel tail")
print("-" * 70)
display(df_final.tail())

# ------------------------------------------------------------------------------
# 12. Save final monthly panel
# ------------------------------------------------------------------------------

df_final.to_parquet(OUTPUT_PARQUET, index=False)
df_final.to_csv(OUTPUT_CSV, index=False)

# Also save compressed CSV.
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUTPUT_CSV, arcname=OUTPUT_CSV.name)

print("--------------------------------------------------")
print("Final monthly panel saved.")
print(f"Parquet : {OUTPUT_PARQUET}")
print(f"CSV     : {OUTPUT_CSV}")
print(f"ZIP     : {OUTPUT_ZIP}")
print("--------------------------------------------------")